In [2]:
# ============================================================
import pandas as pd
import numpy as np
import logging
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 导入自定义模块
from src.core.database import DatabaseManager
from src.core.data_fetcher import DataFetcher
from src.core.spread_calculator import SpreadCalculator
from src.core.indicators import IndicatorBuilder
from src.core.feature_engineering import FeatureEngineer
from src.core.ml_models import MLModel, SignalGenerator
from src.core.backtest import BacktestEngine, PerformanceAnalyzer
from src.core.visualization import Visualizer

# ============================================================
# 配置日志
# ============================================================
log_dir = Path('logs')
log_dir.mkdir(exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_dir / 'debug_strategy.log', encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

logger.info("="*60)
logger.info("开始运行调试版本策略")
logger.info("="*60)

# ============================================================
# 全局变量初始化
# ============================================================
print("\n[1/9] 初始化模块...")

# 数据库和工具类
db_path = "data/trading_data.db"
db = DatabaseManager(db_path)
fetcher = DataFetcher()
spread_calc = SpreadCalculator(db)
indicator_builder = IndicatorBuilder(db)
feature_engineer = FeatureEngineer(indicator_builder)
visualizer = Visualizer(output_dir="outputs/Coking_Coal&Coke_Results")

# 数据存储容器
price_data = {}          # 存储各品种价格数据
spread_data = {}         # 存储价差数据
macro_data = {}          # 存储宏观数据
fundamental_data = {}    # 存储基本面数据

# 特征和模型
features_df = None       # 合并后的特征DataFrame
X_train = None          # 训练集特征
X_test = None           # 测试集特征
y_train = None          # 训练集标签
y_test = None           # 测试集标签
train_idx = None        # 训练集索引
test_idx = None         # 测试集索引
model = None            # 训练好的模型
selected_features = []  # 选择的特征列表

# 回测结果
signals = None          # 交易信号
equity_curve = None     # 权益曲线
trade_log = None        # 交易日志
performance_report = None  # 绩效报告

print("✓ 模块初始化完成")

START_DATE = "2000-01-01"

INFO:__main__:============================================================
INFO:__main__:开始运行调试版本策略
INFO:__main__:============================================================
INFO:src.core.database:数据表创建完成
INFO:src.core.database:数据库已初始化: data/trading_data.db (timeout=30.0s)



[1/9] 初始化模块...
✓ 模块初始化完成


In [3]:
# ============================================================
# 步骤1.1：获取中国焦煤和焦炭期货数据
# ============================================================
print("\n[2.1/9] 开始获取中国焦煤焦炭螺纹钢期货数据...")
logger.info("\n" + "="*60)
logger.info("步骤1.1：获取中国期货数据")
logger.info("="*60)

# 中国期货品种配置
# 焦煤主力合约代码：JM0（大连商品交易所）
# 焦炭主力合约代码：J0（大连商品交易所）
cn_symbols_config = {
    'JM': 'JM0',    # 焦煤主力合约
    'J': 'J0',       # 焦炭主力合约
    'RB': 'RB0'     # 螺纹钢主力合约
}

# 获取中国期货数据
print("  正在从akshare获取中国期货数据...")
for name, symbol in cn_symbols_config.items():
    print(f"  获取 {name} ({symbol}) 数据...")
    logger.info(f"获取 {name} 数据...")
    

    df = fetcher.fetch_akshare_futures_data(symbol, market='CN')

    # 重命名列为英文（akshare返回的是中文列名）

    df.index = pd.to_datetime(df.index)

    if not df.empty:

        
        df_adjusted = df.copy()
        
        # 保存到数据库
        db.insert_price_data(df_adjusted, name, 'futures')
        
        print(f"  ✓ {name}: {len(df_adjusted)} 条记录")
        logger.info(f"{name} 数据获取成功: {len(df_adjusted)} 条记录")
        
        # 显示数据日期范围
        if hasattr(df_adjusted.index, 'min') and hasattr(df_adjusted.index, 'max'):
            date_range = f"{df_adjusted.index.min()} 至 {df_adjusted.index.max()}"
            print(f"      日期范围: {date_range}")
            print(f"      时区: {df_adjusted.index.tz}")
    else:
        print(f"  ✗ {name} 数据获取失败")
        logger.warning(f"{name} 数据获取失败")
            


print("\n✓ 中国期货数据获取完成")
logger.info("中国期货数据获取完成\n")

INFO:__main__:
INFO:__main__:步骤1.1：获取中国期货数据
INFO:__main__:============================================================
INFO:__main__:获取 JM 数据...



[2.1/9] 开始获取中国焦煤焦炭螺纹钢期货数据...
  正在从akshare获取中国期货数据...
  获取 JM (JM0) 数据...


INFO:src.core.data_fetcher:正在从akshare获取 JM0 的期货数据...
INFO:src.core.data_fetcher:成功获取 3060 条 JM0 的数据
INFO:src.core.database:插入了 0 条新数据，跳过了 3060 条重复数据
INFO:__main__:JM 数据获取成功: 3060 条记录
INFO:__main__:获取 J 数据...
INFO:src.core.data_fetcher:正在从akshare获取 J0 的期货数据...


  ✓ JM: 3060 条记录
      日期范围: 2013-03-22 00:00:00+08:00 至 2025-11-03 00:00:00+08:00
      时区: Asia/Shanghai
  获取 J (J0) 数据...


INFO:src.core.data_fetcher:成功获取 3531 条 J0 的数据
INFO:src.core.database:插入了 0 条新数据，跳过了 3531 条重复数据
INFO:__main__:J 数据获取成功: 3531 条记录
INFO:__main__:获取 RB 数据...
INFO:src.core.data_fetcher:正在从akshare获取 RB0 的期货数据...


  ✓ J: 3531 条记录
      日期范围: 2011-04-15 00:00:00+08:00 至 2025-10-31 00:00:00+08:00
      时区: Asia/Shanghai
  获取 RB (RB0) 数据...


INFO:src.core.data_fetcher:成功获取 4031 条 RB0 的数据
INFO:src.core.database:插入了 0 条新数据，跳过了 4031 条重复数据
INFO:__main__:RB 数据获取成功: 4031 条记录
INFO:__main__:中国期货数据获取完成



  ✓ RB: 4031 条记录
      日期范围: 2009-03-27 00:00:00+08:00 至 2025-11-03 00:00:00+08:00
      时区: Asia/Shanghai

✓ 中国期货数据获取完成


In [4]:
# ============================================================
# 步骤1.2：从数据库提取焦煤和焦炭数据（不使用内存数据）
# ============================================================
print("\n[2.2/9] 开始从数据库提取焦煤和焦炭数据...")
logger.info("\n" + "="*60)
logger.info("步骤1.2：从数据库提取中国期货数据")
logger.info("="*60)

# ✅ 清空内存中的数据，强制从数据库读取
print("  清空内存中的临时数据...")
price_data.clear()

# 1. 提取焦煤和焦炭期货价格数据
print("  [1.2.1] 从数据库提取焦煤和焦炭期货价格数据...")
cn_symbols = ['JM', 'J']

for symbol in cn_symbols:
    try:
        df = db.get_price_data(
            symbol=symbol,
            start_date=START_DATE,
            columns=['date', 'open', 'high', 'low', 'close', 'volume']
        )
        
        if not df.empty:
            # 设置索引
            if 'date' in df.columns:
                df.set_index('date', inplace=True)
            
            price_data[symbol] = df
            date_range = f"{df.index.min()} 至 {df.index.max()}"
            print(f"    ✓ {symbol}: {len(df)} 条记录, 日期范围: {date_range}")
            logger.info(f"{symbol} 数据提取成功: {len(df)} 条记录")
        else:
            print(f"    ✗ {symbol}: 数据库中无数据")
            logger.warning(f"{symbol} 数据库中无数据，请先运行数据获取步骤")
            
    except Exception as e:
        print(f"    ✗ {symbol}: 提取失败 - {e}")
        logger.error(f"{symbol} 提取失败: {e}")

# 2. 数据完整性检查
print("\n  [1.2.2] 数据完整性检查...")
required_symbols = ['JM', 'J']
missing_symbols = [s for s in required_symbols if s not in price_data or price_data[s].empty]

if missing_symbols:
    error_msg = f"缺少必要的期货数据: {missing_symbols}"
    print(f"    ❌ {error_msg}")
    logger.error(error_msg)
    print("\n    💡 解决方案:")
    print("    1. 运行上一个代码单元格获取数据")
    print("    2. 检查网络连接和akshare库是否正常")
    print("    3. 查看日志文件: logs/debug_strategy.log")
else:
    print("    ✅ 焦煤和焦炭数据已就绪")
    
    # 显示数据统计信息
    print("\n  [1.2.3] 数据统计信息:")
    for symbol in required_symbols:
        if symbol in price_data:
            df = price_data[symbol]
            print(f"    {symbol}:")
            print(f"      数据量: {len(df)} 条")
            print(f"      日期范围: {df.index.min()} 至 {df.index.max()}")
            print(f"      时区信息: {df.index.tz}")
            print(f"      价格范围: {df['close'].min():.2f} - {df['close'].max():.2f}")
            print(f"      平均成交量: {df['volume'].mean():.0f}")

print("\n✓ 焦煤和焦炭数据提取完成（数据来源：数据库）")
logger.info("焦煤和焦炭数据提取完成\n")

INFO:__main__:
INFO:__main__:步骤1.2：从数据库提取中国期货数据
INFO:__main__:============================================================
INFO:__main__:JM 数据提取成功: 3060 条记录
INFO:__main__:J 数据提取成功: 3531 条记录
INFO:__main__:焦煤和焦炭数据提取完成




[2.2/9] 开始从数据库提取焦煤和焦炭数据...
  清空内存中的临时数据...
  [1.2.1] 从数据库提取焦煤和焦炭期货价格数据...
    ✓ JM: 3060 条记录, 日期范围: 2013-03-22 00:00:00+08:00 至 2025-11-03 00:00:00+08:00
    ✓ J: 3531 条记录, 日期范围: 2011-04-15 00:00:00+08:00 至 2025-10-31 00:00:00+08:00

  [1.2.2] 数据完整性检查...
    ✅ 焦煤和焦炭数据已就绪

  [1.2.3] 数据统计信息:
    JM:
      数据量: 3060 条
      日期范围: 2013-03-22 00:00:00+08:00 至 2025-11-03 00:00:00+08:00
      时区信息: pytz.FixedOffset(480)
      价格范围: 494.00 - 3847.00
      平均成交量: 253461
    J:
      数据量: 3531 条
      日期范围: 2011-04-15 00:00:00+08:00 至 2025-10-31 00:00:00+08:00
      时区信息: pytz.FixedOffset(480)
      价格范围: 606.50 - 4402.00
      平均成交量: 262075

✓ 焦煤和焦炭数据提取完成（数据来源：数据库）


In [5]:
# 获取宏观数据
print("\n  获取宏观经济数据...")
logger.info("获取宏观经济数据...")

# VIX波动率指数
vix_data = fetcher.fetch_index_data('VIX', start_date=START_DATE)
if not vix_data.empty:
    db.insert_price_data(vix_data, 'VIX', 'index')
    macro_data['VIX'] = vix_data
    print(f"  ✓ VIX: {len(vix_data)} 条记录")
    logger.info(f"VIX数据获取成功: {len(vix_data)} 条记录")

# 美元指数
dxy_data = fetcher.fetch_index_data('DXY', start_date=START_DATE)
if not dxy_data.empty:
    db.insert_price_data(dxy_data, 'DXY', 'index')
    macro_data['DXY'] = dxy_data
    print(f"  ✓ DXY: {len(dxy_data)} 条记录")
    logger.info(f"DXY数据获取成功: {len(dxy_data)} 条记录")

INFO:__main__:获取宏观经济数据...



  获取宏观经济数据...


INFO:src.core.data_fetcher:正在从yfinance获取 ^VIX 的数据...
INFO:src.core.data_fetcher:成功获取 6498 条 ^VIX 的数据
INFO:src.core.database:插入了 0 条新数据，跳过了 6498 条重复数据
INFO:__main__:VIX数据获取成功: 6498 条记录
INFO:src.core.data_fetcher:正在从yfinance获取 DX-Y.NYB 的数据...
INFO:src.core.data_fetcher:成功获取 6527 条 DX-Y.NYB 的数据


  ✓ VIX: 6498 条记录


INFO:src.core.database:插入了 0 条新数据，跳过了 6527 条重复数据
INFO:__main__:DXY数据获取成功: 6527 条记录


  ✓ DXY: 6527 条记录


## 🏗️ 步骤1.2.5：获取螺纹钢数据（下游行业指标）

螺纹钢是焦炭和焦煤的主要下游产品，其价格和产量数据对于分析焦炭焦煤价差具有重要参考价值。

- **品种代码**: RB (螺纹钢主力合约)
- **市场**: 上海期货交易所
- **数据类型**: 行业下游数据
- **作用**: 反映钢铁行业需求，影响焦炭焦煤需求

## 📊 步骤1.2.6：从数据库提取螺纹钢数据

从数据库中提取螺纹钢数据，作为宏观/行业下游指标使用

In [6]:
# ============================================================
# 步骤1.2.6：从数据库提取螺纹钢数据（按宏观数据方式处理）
# ============================================================
print("\n[2.2.6/9] 从数据库提取螺纹钢数据（下游行业指标）...")
logger.info("\n" + "="*60)
logger.info("步骤1.2.6：提取螺纹钢数据")
logger.info("="*60)

# 提取螺纹钢数据（作为宏观/行业指标处理）
print("  [1.2.6.1] 提取螺纹钢价格数据...")
downstream_symbols = ['RB']

for symbol in downstream_symbols:
    try:
        df = db.get_price_data(
            symbol=symbol,
            start_date=START_DATE,
            columns=['date', 'close', 'volume', 'open', 'high', 'low']
        )
        
        if not df.empty:
            # 设置索引
            if 'date' in df.columns:
                df.set_index('date', inplace=True)
            
            # 确保索引有时区（与宏观数据保持一致）
            if hasattr(df.index, 'tz'):
                if df.index.tz is None:
                    df.index = df.index.tz_localize('UTC')
                    print(f"    ⚠️  {symbol}: 数据库数据无时区，已添加UTC时区")
                else:
                    print(f"    ✓ {symbol}: 时区 = {df.index.tz}")
            
            # 存储到 macro_data 字典（作为行业下游宏观指标）
            macro_data[symbol] = df
            
            date_range = f"{df.index.min()} 至 {df.index.max()}"
            print(f"    ✓ {symbol}: {len(df)} 条记录, 日期范围: {date_range}")
            logger.info(f"{symbol} 数据提取成功: {len(df)} 条记录")
            
            # 显示统计信息
            print(f"      价格范围: {df['close'].min():.2f} - {df['close'].max():.2f}")
            print(f"      平均成交量: {df['volume'].mean():.0f}")
            
        else:
            print(f"    ✗ {symbol}: 数据库中无数据")
            logger.warning(f"{symbol} 数据库中无数据，请先运行数据获取步骤")
            
    except Exception as e:
        print(f"    ✗ {symbol}: 提取失败 - {e}")
        logger.error(f"{symbol} 提取失败: {e}")

# 数据验证
print("\n  [1.2.6.2] 螺纹钢数据验证...")
if 'RB' in macro_data and not macro_data['RB'].empty:
    rb_df = macro_data['RB']
    print(f"    ✅ 螺纹钢数据已就绪")
    print(f"    数据量: {len(rb_df)} 条")
    print(f"    日期范围: {rb_df.index.min()} 至 {rb_df.index.max()}")
    print(f"    时区信息: {rb_df.index.tz}")
    
    # 计算与焦炭、焦煤的相关性（如果数据存在）
    if 'JM' in price_data and 'J' in price_data:
        print(f"\n  [1.2.6.3] 相关性分析:")
        # 对齐数据进行相关性分析
        rb_close = rb_df['close']
        
        if 'JM' in price_data:
            jm_close = price_data['JM']['close']
            # 找到共同的日期索引
            common_dates = rb_close.index.intersection(jm_close.index)
            if len(common_dates) > 0:
                corr_jm = rb_close.loc[common_dates].corr(jm_close.loc[common_dates])
                print(f"      螺纹钢 vs 焦煤: {corr_jm:.4f}")
        
        if 'J' in price_data:
            j_close = price_data['J']['close']
            common_dates = rb_close.index.intersection(j_close.index)
            if len(common_dates) > 0:
                corr_j = rb_close.loc[common_dates].corr(j_close.loc[common_dates])
                print(f"      螺纹钢 vs 焦炭: {corr_j:.4f}")
else:
    print(f"    ⚠️  螺纹钢数据未就绪")
    print(f"    💡 请先运行上一个单元格获取数据")

print("\n✓ 螺纹钢数据提取完成（数据来源：数据库，存储位置：macro_data['RB']）")
logger.info("螺纹钢数据提取完成\n")

INFO:__main__:
INFO:__main__:步骤1.2.6：提取螺纹钢数据
INFO:__main__:============================================================
INFO:__main__:RB 数据提取成功: 4031 条记录
INFO:__main__:螺纹钢数据提取完成




[2.2.6/9] 从数据库提取螺纹钢数据（下游行业指标）...
  [1.2.6.1] 提取螺纹钢价格数据...
    ✓ RB: 时区 = pytz.FixedOffset(480)
    ✓ RB: 4031 条记录, 日期范围: 2009-03-27 00:00:00+08:00 至 2025-11-03 00:00:00+08:00
      价格范围: 1626.00 - 6171.00
      平均成交量: 2530897

  [1.2.6.2] 螺纹钢数据验证...
    ✅ 螺纹钢数据已就绪
    数据量: 4031 条
    日期范围: 2009-03-27 00:00:00+08:00 至 2025-11-03 00:00:00+08:00
    时区信息: pytz.FixedOffset(480)

  [1.2.6.3] 相关性分析:
      螺纹钢 vs 焦煤: 0.8294
      螺纹钢 vs 焦炭: 0.8539

✓ 螺纹钢数据提取完成（数据来源：数据库，存储位置：macro_data['RB']）


## 📦 步骤1.3：导入焦炭焦煤仓单和库存数据

从外部CSV文件导入基本面数据：
- **仓单数量**：焦炭、焦煤的仓单数量
- **港口库存**：环渤海港口库存量
- **铁路调入量**：环渤海港口铁路调入量

数据将存入数据库的 `fundamental_data` 表，数据源标识为 `COAL_INVENTORY`

In [7]:
# ============================================================
# 步骤1.3：导入焦炭焦煤仓单和库存数据到数据库
# ============================================================
print("\n[2.3/9] 导入焦炭焦煤仓单和库存数据...")
logger.info("\n" + "="*60)
logger.info("步骤1.3：导入基本面数据")
logger.info("="*60)

# 导入处理函数
import sys
sys.path.insert(0, str(Path.cwd()))

# 文件路径
csv_path = "outside_data_import/仓单数量焦炭等_20251103_090504.csv"

try:
    print("  [1.3.1] 读取CSV文件...")
    # 读取CSV
    inventory_df = pd.read_csv(csv_path)
    print(f"    ✓ 读取 {len(inventory_df)} 行数据")
    
    # 重命名列为英文和Python合法名称
    print("  [1.3.2] 重命名列...")
    column_mapping = {
        '指标名称': 'date',
        '仓单数量:焦炭': 'coke_warehouse_receipt',
        '仓单数量:焦煤': 'coking_coal_warehouse_receipt',
        '港口库存:环渤海港': 'bohai_port_inventory',
        '港口铁路调入量:环渤海港': 'bohai_port_rail_inflow'
    }
    inventory_df = inventory_df.rename(columns=column_mapping)
    print(f"    ✓ 新列名: {list(inventory_df.columns)}")
    
    # 处理日期和时区（东八区）
    print("  [1.3.3] 处理日期和时区（东八区）...")
    inventory_df['date'] = pd.to_datetime(inventory_df['date'])
    inventory_df['date'] = inventory_df['date'].dt.tz_localize('Asia/Shanghai')
    print(f"    ✓ 时区: {inventory_df['date'].dt.tz}")
    print(f"    ✓ 日期范围: {inventory_df['date'].min()} 至 {inventory_df['date'].max()}")
    
    # 清理和转换数值列
    print("  [1.3.4] 清理数值列...")
    numeric_columns = [
        'coke_warehouse_receipt',
        'coking_coal_warehouse_receipt', 
        'bohai_port_inventory',
        'bohai_port_rail_inflow'
    ]
    
    for col in numeric_columns:
        inventory_df[col] = inventory_df[col].astype(str).str.strip()
        inventory_df[col] = inventory_df[col].replace('', pd.NA).replace('nan', pd.NA)
        inventory_df[col] = pd.to_numeric(inventory_df[col], errors='coerce')
        non_null = inventory_df[col].notna().sum()
        print(f"      {col}: {non_null} 个有效值")
    
    # 设置索引
    inventory_df.set_index('date', inplace=True)
    
    # 存入数据库
    print("  [1.3.5] 存入数据库...")
    success_count = 0
    error_count = 0
    
    for date, row in inventory_df.iterrows():
        try:
            # 准备数据字典（只包含非空值）
            data_dict = {}
            for col in numeric_columns:
                if pd.notna(row[col]):
                    data_dict[col] = float(row[col])
            
            # 只有当有数据时才插入
            if data_dict:
                db.insert_fundamental_data(
                    data_source='COAL_INVENTORY',
                    report_date=date.strftime('%Y-%m-%d'),
                    publish_date=date.strftime('%Y-%m-%d'),  # 修复：使用publish_date而不是period_end
                    data_dict=data_dict  # 修复：使用data_dict而不是data
                )
                success_count += 1
        except Exception as e:
            error_count += 1
            if error_count <= 5:  # 只显示前5个错误
                print(f"      ✗ {date}: {e}")
    
    print(f"    ✓ 成功插入 {success_count} 条记录")
    if error_count > 0:
        print(f"    ⚠️  失败 {error_count} 条记录")
    
    # 验证数据
    print("  [1.3.6] 验证数据...")
    verify_df = db.get_fundamental_data(
        data_source='COAL_INVENTORY',
        start_date='2002-01-01'
    )
    
    if not verify_df.empty:
        print(f"    ✓ 数据库中共有 {len(verify_df)} 条 COAL_INVENTORY 记录")
        print(f"    ✓ 日期范围: {verify_df['report_date'].min()} 至 {verify_df['report_date'].max()}")
    else:
        print("    ⚠️  数据库中无 COAL_INVENTORY 数据")
    
    print("\n✓ 焦炭焦煤仓单和库存数据导入完成")
    logger.info("焦炭焦煤仓单和库存数据导入完成\n")
    
except FileNotFoundError:
    print(f"  ✗ 文件不存在: {csv_path}")
    logger.error(f"文件不存在: {csv_path}")
except Exception as e:
    print(f"  ✗ 导入失败: {e}")
    logger.error(f"导入失败: {e}")
    import traceback
    traceback.print_exc()


INFO:__main__:
INFO:__main__:步骤1.3：导入基本面数据
INFO:__main__:============================================================



[2.3/9] 导入焦炭焦煤仓单和库存数据...
  [1.3.1] 读取CSV文件...
    ✓ 读取 6659 行数据
  [1.3.2] 重命名列...
    ✓ 新列名: ['date', 'coke_warehouse_receipt', 'coking_coal_warehouse_receipt', 'bohai_port_inventory', 'bohai_port_rail_inflow']
  [1.3.3] 处理日期和时区（东八区）...
    ✓ 时区: Asia/Shanghai
    ✓ 日期范围: 2002-01-23 00:00:00+08:00 至 2025-10-31 00:00:00+08:00
  [1.3.4] 清理数值列...
      coke_warehouse_receipt: 2175 个有效值
      coking_coal_warehouse_receipt: 490 个有效值
      bohai_port_inventory: 6641 个有效值
      bohai_port_rail_inflow: 6245 个有效值
  [1.3.5] 存入数据库...


INFO:__main__:焦炭焦煤仓单和库存数据导入完成



    ✓ 成功插入 6659 条记录
  [1.3.6] 验证数据...
    ✓ 数据库中共有 6659 条 COAL_INVENTORY 记录
    ✓ 日期范围: 2002-01-23 00:00:00 至 2025-10-31 00:00:00

✓ 焦炭焦煤仓单和库存数据导入完成


## 📊 步骤1.4：从数据库提取焦炭焦煤基本面数据

从数据库提取仓单和库存数据作为基本面指标

In [8]:
# ============================================================
# 步骤1.4：从数据库提取焦炭焦煤基本面数据
# ============================================================
print("\n[2.4/9] 从数据库提取焦炭焦煤基本面数据...")
logger.info("\n" + "="*60)
logger.info("步骤1.4：提取基本面数据")
logger.info("="*60)

try:
    print("  [1.4.1] 提取 COAL_INVENTORY 基本面数据...")
    inventory_fundamental = db.get_fundamental_data(
        data_source='COAL_INVENTORY',
        start_date=START_DATE
    )
    
    if not inventory_fundamental.empty:
        # 解析JSON数据并转换为DataFrame
        import json
        
        print("  [1.4.2] 解析基本面数据...")
        data_list = []
        for _, row in inventory_fundamental.iterrows():
            data_dict = json.loads(row['data_json'])
            data_dict['report_date'] = row['report_date']
            data_list.append(data_dict)
        
        # 创建DataFrame
        inventory_df = pd.DataFrame(data_list)
        inventory_df['report_date'] = pd.to_datetime(inventory_df['report_date'])
        
        # 添加东八区时区
        inventory_df['report_date'] = inventory_df['report_date'].dt.tz_localize('Asia/Shanghai')
        
        # 设置索引
        inventory_df.set_index('report_date', inplace=True)
        
        # 存入fundamental_data字典
        fundamental_data['COAL_INVENTORY'] = inventory_df
        
        print(f"    ✓ 提取 {len(inventory_df)} 条基本面数据")
        print(f"    ✓ 日期范围: {inventory_df.index.min()} 至 {inventory_df.index.max()}")
        print(f"    ✓ 时区: {inventory_df.index.tz}")
        
        # 显示数据列
        print(f"    ✓ 数据字段:")
        for col in inventory_df.columns:
            non_null = inventory_df[col].notna().sum()
            print(f"      - {col}: {non_null} 个有效值")
        
        # 显示数据样例
        print("\n  [1.4.3] 数据样例（最近10条）:")
        print(inventory_df.tail(10).to_string())
        
    else:
        print("    ⚠️  数据库中无 COAL_INVENTORY 数据")
        print("    💡 请先运行上一个单元格导入数据")
        logger.warning("COAL_INVENTORY 数据库中无数据")
        
except Exception as e:
    print(f"    ✗ 提取失败: {e}")
    logger.error(f"COAL_INVENTORY 提取失败: {e}")
    import traceback
    traceback.print_exc()

print("\n✓ 焦炭焦煤基本面数据提取完成")
logger.info("焦炭焦煤基本面数据提取完成\n")

INFO:__main__:
INFO:__main__:步骤1.4：提取基本面数据
INFO:__main__:============================================================



[2.4/9] 从数据库提取焦炭焦煤基本面数据...
  [1.4.1] 提取 COAL_INVENTORY 基本面数据...
  [1.4.2] 解析基本面数据...


INFO:__main__:焦炭焦煤基本面数据提取完成



    ✓ 提取 6659 条基本面数据
    ✓ 日期范围: 2002-01-23 00:00:00+08:00 至 2025-10-31 00:00:00+08:00
    ✓ 时区: Asia/Shanghai
    ✓ 数据字段:
      - bohai_port_inventory: 6641 个有效值
      - bohai_port_rail_inflow: 6245 个有效值
      - coke_warehouse_receipt: 2175 个有效值
      - coking_coal_warehouse_receipt: 490 个有效值

  [1.4.3] 数据样例（最近10条）:
                           bohai_port_inventory  bohai_port_rail_inflow  coke_warehouse_receipt  coking_coal_warehouse_receipt
report_date                                                                                                                   
2025-10-22 00:00:00+08:00                1840.6                   167.1                  2070.0                          200.0
2025-10-23 00:00:00+08:00                1850.0                   170.8                  2070.0                          100.0
2025-10-24 00:00:00+08:00                1846.9                   176.2                  2070.0                          100.0
2025-10-25 00:00:00+08:00                1832.

In [9]:
fundamental_data["COAL_INVENTORY"]["coke_warehouse_receipt"].loc["2011-08-24":] = \
    fundamental_data["COAL_INVENTORY"]["coke_warehouse_receipt"].loc["2011-08-24":].fillna(0)

fundamental_data["COAL_INVENTORY"]["coking_coal_warehouse_receipt"].loc["2013-09-12":] = \
    fundamental_data["COAL_INVENTORY"]["coking_coal_warehouse_receipt"].loc["2013-09-12":].fillna(0)

In [10]:
print("\n  [1.2] 提取宏观数据...")
macro_symbols = ['VIX', 'DXY']

for symbol in macro_symbols:
    try:
        df = db.get_price_data(
            symbol=symbol,
            start_date=START_DATE,
            columns=['date', 'close']
        )
        
        if not df.empty:
            # 设置索引
            if 'date' in df.columns:
                df.set_index('date', inplace=True)
            
            # 确保索引无时区
            # if hasattr(df.index, 'tz') and df.index.tz is not None:
            #     df.index = df.index.tz_localize(None)
            
            macro_data[symbol] = df
            print(f"    ✓ {symbol}: {len(df)} 条记录")
            logger.info(f"{symbol} 数据提取成功: {len(df)} 条记录")
        else:
            print(f"    ✗ {symbol}: 数据库中无数据")
            logger.warning(f"{symbol} 数据库中无数据")
            
    except Exception as e:
        print(f"    ✗ {symbol}: 提取失败 - {e}")
        logger.error(f"{symbol} 提取失败: {e}")


  [1.2] 提取宏观数据...


INFO:__main__:VIX 数据提取成功: 6498 条记录


    ✓ VIX: 6498 条记录


INFO:__main__:DXY 数据提取成功: 6528 条记录


    ✓ DXY: 6528 条记录


In [11]:

# 4. 数据验证
print("\n  [1.4] 数据验证...")
print("    期货数据:")
for symbol, df in price_data.items():
    date_range = f"{df.index.min()} 至 {df.index.max()}"
    print(f"      {symbol}: {len(df)} 条, 日期范围: {date_range}")

print("    宏观数据:")
for symbol, df in macro_data.items():
    date_range = f"{df.index.min()} 至 {df.index.max()}"
    print(f"      {symbol}: {len(df)} 条, 日期范围: {date_range}")

# if 'EIA' in fundamental_data:
#     eia = fundamental_data['EIA']
#     date_range = f"{eia.index.min()} 至 {eia.index.max()}"
#     print(f"    基本面数据:")
#     print(f"      EIA: {len(eia)} 条, 日期范围: {date_range}")

# 5. 检查数据完整性
print("\n  [1.5] 数据完整性检查...")
required_symbols = ['JM', 'J']
missing_symbols = [s for s in required_symbols if s not in price_data or price_data[s].empty]

if missing_symbols:
    error_msg = f"缺少必要的期货数据: {missing_symbols}"
    print(f"    ❌ {error_msg}")
    logger.error(error_msg)
    print("\n    💡 解决方案:")
    print("    1. 运行数据更新脚本: python scripts/daily_data_update.py")
    print("    2. 检查网络连接和数据源可用性")
    print("    3. 查看日志文件: logs/daily_update.log")
    raise ValueError(error_msg)
else:
    print("    ✅ 所有必要数据已就绪")

print("\n✓ 数据提取完成")
logger.info("数据提取完成\n")



# 🔍 调试点1：在此处设置断点，检查 price_data, macro_data 的内容
# 可以在调试控制台输入: price_data.keys(), len(price_data['CL'])

INFO:__main__:数据提取完成




  [1.4] 数据验证...
    期货数据:
      JM: 3060 条, 日期范围: 2013-03-22 00:00:00+08:00 至 2025-11-03 00:00:00+08:00
      J: 3531 条, 日期范围: 2011-04-15 00:00:00+08:00 至 2025-10-31 00:00:00+08:00
    宏观数据:
      VIX: 6498 条, 日期范围: 2000-01-03 00:00:00-06:00 至 2025-10-31 00:00:00-05:00
      DXY: 6528 条, 日期范围: 2000-01-03 00:00:00-05:00 至 2025-11-02 00:00:00-04:00
      RB: 4031 条, 日期范围: 2009-03-27 00:00:00+08:00 至 2025-11-03 00:00:00+08:00

  [1.5] 数据完整性检查...
    ✅ 所有必要数据已就绪

✓ 数据提取完成


## 📊 步骤2.1：计算焦煤焦炭利润价差

使用**利润价差法**（虚拟焦化厂利润）计算焦煤焦炭价差：

### 公式
```
盘面炼焦利润 = (1 × 焦炭价格) - (K × 焦煤价格)
```

### 配比系数 K
- **理论值**: 1.35（生产1吨焦炭需要1.35吨焦煤）
- **实际范围**: 1.33-1.38（根据焦炭和焦煤具体指标微调）

### 计算示例
```
焦炭价格 (J)  = 2200 元/吨
焦煤价格 (JM) = 1500 元/吨
配比系数 (K)  = 1.35

盘面利润 = 2200 - (1.35 × 1500) = 2200 - 2025 = 175 元/吨
```

**意义**：
- 利润为正：焦化厂盈利，可能增产
- 利润为负：焦化厂亏损，可能减产
- 利润变化：预示供需关系变化

In [12]:
# ============================================================
# 步骤2.1：计算焦煤焦炭利润价差
# ============================================================
print("\n[3.1/9] 计算焦煤焦炭利润价差...")
logger.info("\n" + "="*60)
logger.info("步骤2.1：焦煤焦炭利润价差")
logger.info("="*60)

# 检查焦煤和焦炭数据是否存在
if 'JM' in price_data and 'J' in price_data:
    print("  [2.1.1] 配置价差参数...")
    
    # 配比系数：生产1吨焦炭需要的焦煤吨数
    # 理论值：1.35（可根据实际情况在1.33-1.38之间调整）
    COKING_RATIO = 1.3
    
    print(f"    ✓ 配比系数 K = {COKING_RATIO}")
    print(f"    说明：生产1吨焦炭需要 {COKING_RATIO} 吨焦煤")
    
    # 注册焦煤焦炭价差配置
    spread_calc.create_spread(
        spread_name='COKING_PROFIT',
        components={
            'J': 1.0,        # 焦炭价格系数
            'JM': -COKING_RATIO  # 焦煤价格系数（负数表示成本）
        }.items(),
        save_config=True
    )
    
    print("  [2.1.2] 计算价差...")
    # 计算价差
    coking_spread_df = spread_calc.calculate_spread(
        'COKING_PROFIT',
        price_data,
        price_column='close'
    )
    
    # 添加统计特征
    coking_spread_df = spread_calc.get_spread_statistics(coking_spread_df, window=20)
    spread_data['COKING_PROFIT'] = coking_spread_df
    
    print(f"    ✓ 价差数据点: {len(coking_spread_df)}")
    print(f"    ✓ 时间范围: {coking_spread_df.index.min()} 至 {coking_spread_df.index.max()}")
    
    # 价差统计信息
    print(f"\n  [2.1.3] 价差统计:")
    print(f"    均值: {coking_spread_df['spread'].mean():.2f} 元/吨")
    print(f"    标准差: {coking_spread_df['spread'].std():.2f} 元/吨")
    print(f"    最大值: {coking_spread_df['spread'].max():.2f} 元/吨")
    print(f"    最小值: {coking_spread_df['spread'].min():.2f} 元/吨")
    print(f"    当前值: {coking_spread_df['spread'].iloc[-1]:.2f} 元/吨")
    
    # 分析盈亏情况
    profitable_days = (coking_spread_df['spread'] > 0).sum()
    total_days = len(coking_spread_df)
    profit_ratio = profitable_days / total_days * 100
    
    print(f"\n  [2.1.4] 盈亏分析:")
    print(f"    盈利天数: {profitable_days} 天 ({profit_ratio:.1f}%)")
    print(f"    亏损天数: {total_days - profitable_days} 天 ({100-profit_ratio:.1f}%)")
    
    if coking_spread_df['spread'].iloc[-1] > 0:
        print(f"    当前状态: 💰 盈利 {coking_spread_df['spread'].iloc[-1]:.2f} 元/吨")
    else:
        print(f"    当前状态: 📉 亏损 {abs(coking_spread_df['spread'].iloc[-1]):.2f} 元/吨")
    
    # 保存到数据库
    print("\n  [2.1.5] 保存到数据库...")
    db.insert_indicator_data(
        'COKING_PROFIT',
        coking_spread_df[['spread']],
        metadata={
            'type': 'coking_spread',
            'ratio': f'1:-{COKING_RATIO}',
            'description': f'焦化厂盘面利润',
            'unit': '元/吨'
        }
    )
    print("    ✓ 数据已保存")
    
    # 平稳性检验
    print("\n  [2.1.6] 进行ADF平稳性检验...")
    indicator_builder.test_stationarity(
        coking_spread_df['spread'],
        name='COKING_PROFIT Spread'
    )
    
    logger.info(f"焦煤焦炭利润价差计算完成: {len(coking_spread_df)} 个数据点")
    
else:
    print("  ✗ 缺少焦煤(JM)或焦炭(J)价格数据")
    if 'JM' not in price_data:
        print("    缺少: 焦煤(JM)")
    if 'J' not in price_data:
        print("    缺少: 焦炭(J)")
    logger.error("缺少计算焦煤焦炭价差所需的价格数据")

print("\n✓ 焦煤焦炭利润价差计算完成")
logger.info("焦煤焦炭利润价差计算完成\n")

INFO:__main__:
INFO:__main__:步骤2.1：焦煤焦炭利润价差
INFO:__main__:============================================================
ERROR:src.core.spread_calculator:保存价差配置失败: Object of type dict_items is not JSON serializable
INFO:src.core.spread_calculator:创建价差配置: COKING_PROFIT - 多头(1.0xJ) - 空头(1.3xJM)
INFO:src.core.spread_calculator:计算价差 COKING_PROFIT: 3059 个数据点



[3.1/9] 计算焦煤焦炭利润价差...
  [2.1.1] 配置价差参数...
    ✓ 配比系数 K = 1.3
    说明：生产1吨焦炭需要 1.3 吨焦煤
  [2.1.2] 计算价差...


INFO:src.core.spread_calculator:价差统计特征计算完成，窗口: 20


    ✓ 价差数据点: 3059
    ✓ 时间范围: 2013-03-22 00:00:00+08:00 至 2025-10-31 00:00:00+08:00

  [2.1.3] 价差统计:
    均值: 190.23 元/吨
    标准差: 198.92 元/吨
    最大值: 1009.40 元/吨
    最小值: -726.65 元/吨
    当前值: 105.20 元/吨

  [2.1.4] 盈亏分析:
    盈利天数: 2672 天 (87.3%)
    亏损天数: 387 天 (12.7%)
    当前状态: 💰 盈利 105.20 元/吨

  [2.1.5] 保存到数据库...


INFO:src.core.database:插入了 0 条新指标数据
INFO:src.core.indicators:
INFO:src.core.indicators:ADF平稳性检验结果 - COKING_PROFIT Spread
INFO:src.core.indicators:==================================================
INFO:src.core.indicators:ADF统计量: -4.029681
INFO:src.core.indicators:P值: 0.001263
INFO:src.core.indicators:使用滞后阶数: 6
INFO:src.core.indicators:观测值数量: 3052
INFO:src.core.indicators:临界值:
INFO:src.core.indicators:  1%: -3.432494
INFO:src.core.indicators:  5%: -2.862487
INFO:src.core.indicators:  10%: -2.567274
INFO:src.core.indicators:结论: COKING_PROFIT Spread 是平稳序列 (p < 0.05)
INFO:src.core.indicators:==================================================

INFO:__main__:焦煤焦炭利润价差计算完成: 3059 个数据点
INFO:__main__:焦煤焦炭利润价差计算完成



    ✓ 数据已保存

  [2.1.6] 进行ADF平稳性检验...

✓ 焦煤焦炭利润价差计算完成


### 📈 可视化焦煤焦炭价差

查看盘面利润的历史变化趋势

In [13]:
# 可视化焦煤焦炭价差
if 'COKING_PROFIT' in spread_data:
    import matplotlib.pyplot as plt
    import matplotlib.dates as mdates
    
    coking_df = spread_data['COKING_PROFIT']
    
    # 创建图表
    fig, axes = plt.subplots(3, 1, figsize=(15, 12))
    
    # 子图1：价差时间序列
    ax1 = axes[0]
    ax1.plot(coking_df.index, coking_df['spread'], linewidth=1, color='navy', alpha=0.7)
    ax1.axhline(y=0, color='red', linestyle='--', linewidth=1, label='盈亏平衡线')
    ax1.fill_between(coking_df.index, 0, coking_df['spread'], 
                      where=(coking_df['spread'] > 0), color='green', alpha=0.2, label='盈利区域')
    ax1.fill_between(coking_df.index, 0, coking_df['spread'], 
                      where=(coking_df['spread'] <= 0), color='red', alpha=0.2, label='亏损区域')
    ax1.set_title('焦化厂盘面利润时间序列 (1×焦炭 - 1.35×焦煤)', fontsize=14, fontweight='bold')
    ax1.set_ylabel('利润 (元/吨)', fontsize=12)
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc='best')
    
    # 子图2：价差分布直方图
    ax2 = axes[1]
    ax2.hist(coking_df['spread'], bins=50, color='steelblue', alpha=0.7, edgecolor='black')
    ax2.axvline(x=0, color='red', linestyle='--', linewidth=2, label='盈亏平衡点')
    ax2.axvline(x=coking_df['spread'].mean(), color='green', linestyle='--', linewidth=2, 
                label=f'均值: {coking_df["spread"].mean():.2f}')
    ax2.set_title('盘面利润分布', fontsize=14, fontweight='bold')
    ax2.set_xlabel('利润 (元/吨)', fontsize=12)
    ax2.set_ylabel('频数', fontsize=12)
    ax2.legend(loc='best')
    ax2.grid(True, alpha=0.3, axis='y')
    
    # 子图3：滚动统计
    ax3 = axes[2]
    # 计算滚动均值和标准差
    rolling_mean = coking_df['spread'].rolling(window=60).mean()
    rolling_std = coking_df['spread'].rolling(window=60).std()
    
    ax3.plot(coking_df.index, coking_df['spread'], linewidth=1, color='lightgray', alpha=0.5, label='原始价差')
    ax3.plot(rolling_mean.index, rolling_mean, linewidth=2, color='blue', label='60日均值')
    ax3.fill_between(rolling_mean.index, 
                      rolling_mean - 2*rolling_std, 
                      rolling_mean + 2*rolling_std,
                      alpha=0.2, color='blue', label='±2标准差区间')
    ax3.axhline(y=0, color='red', linestyle='--', linewidth=1)
    ax3.set_title('60日滚动均值与波动区间', fontsize=14, fontweight='bold')
    ax3.set_xlabel('日期', fontsize=12)
    ax3.set_ylabel('利润 (元/吨)', fontsize=12)
    ax3.grid(True, alpha=0.3)
    ax3.legend(loc='best')
    
    plt.tight_layout()
    
    # 保存图表
    output_path = 'outputs/Coking_Coal&Coke_Results/coking_profit_spread_analysis.png'
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"✓ 图表已保存: {output_path}")
    
    plt.show()
    
    # 打印关键统计信息
    print("\n" + "="*60)
    print("焦化厂盘面利润统计摘要")
    print("="*60)
    print(f"配比系数: 1.35 (1吨焦炭需要1.35吨焦煤)")
    print(f"数据周期: {coking_df.index.min().strftime('%Y-%m-%d')} 至 {coking_df.index.max().strftime('%Y-%m-%d')}")
    print(f"\n利润统计:")
    print(f"  平均利润: {coking_df['spread'].mean():.2f} 元/吨")
    print(f"  中位数:   {coking_df['spread'].median():.2f} 元/吨")
    print(f"  标准差:   {coking_df['spread'].std():.2f} 元/吨")
    print(f"  最大利润: {coking_df['spread'].max():.2f} 元/吨 ({coking_df['spread'].idxmax().strftime('%Y-%m-%d')})")
    print(f"  最小利润: {coking_df['spread'].min():.2f} 元/吨 ({coking_df['spread'].idxmin().strftime('%Y-%m-%d')})")
    print(f"  当前利润: {coking_df['spread'].iloc[-1]:.2f} 元/吨")
    print(f"\n盈亏分析:")
    profitable = (coking_df['spread'] > 0).sum()
    total = len(coking_df)
    print(f"  盈利天数: {profitable} ({profitable/total*100:.1f}%)")
    print(f"  亏损天数: {total-profitable} ({(total-profitable)/total*100:.1f}%)")
    print("="*60)
else:
    print("⚠️  未找到 COKING_PROFIT 价差数据，请先运行价差计算单元格")

✓ 图表已保存: outputs/Coking_Coal&Coke_Results/coking_profit_spread_analysis.png

焦化厂盘面利润统计摘要
配比系数: 1.35 (1吨焦炭需要1.35吨焦煤)
数据周期: 2013-03-22 至 2025-10-31

利润统计:
  平均利润: 190.23 元/吨
  中位数:   149.40 元/吨
  标准差:   198.92 元/吨
  最大利润: 1009.40 元/吨 (2018-08-17)
  最小利润: -726.65 元/吨 (2021-09-30)
  当前利润: 105.20 元/吨

盈亏分析:
  盈利天数: 2672 (87.3%)
  亏损天数: 387 (12.7%)


In [14]:
# ============================================================
# 步骤3：特征工程
# ============================================================
print("\n[4/9] 开始特征工程...")
logger.info("\n" + "="*60)
logger.info("步骤3：特征工程")
logger.info("="*60)

# 获取主价差数据
main_spread = spread_data.get('COKING_PROFIT')
if main_spread is None or main_spread.empty:
    print("  ✗ 价差数据不可用，停止执行")
    logger.error("价差数据不可用")
    raise ValueError("价差数据不可用")

# 1. 创建价差特征
print("  [3.1] 创建价差特征...")
spread_features = feature_engineer.create_spread_features(main_spread)
print(f"    ✓ 价差特征: {len(spread_features.columns)} 个")

# 2. 创建价格特征
print("  [3.2] 创建价格特征...")
price_features = feature_engineer.create_price_features(price_data)
print(f"    ✓ 价格特征: {len(price_features.columns)} 个")

# 3. 创建技术指标特征
print("  [3.3] 创建技术指标特征...")
technical_features = pd.DataFrame(index=main_spread.index)
for symbol, df in price_data.items():
    if symbol in [np.nan]:  # 只对焦煤和焦炭计算技术指标
        tech_df = feature_engineer.create_technical_features(df, symbol)
        # 选择关键列
        key_cols = [col for col in tech_df.columns 
                   if any(x in col for x in ['RSI', 'MACD', 'BB_percent'])]
        if key_cols:
            technical_features = technical_features.join(tech_df[key_cols], how='outer')
print(f"    ✓ 技术指标特征: {len(technical_features.columns)} 个")

# # 4. 创建季节性特征
# print("  [3.4] 创建季节性特征...")
# seasonal_features = feature_engineer.create_seasonal_features(
#     pd.DataFrame(index=main_spread.index)
# )
# print(f"    ✓ 季节性特征: {len(seasonal_features.columns)} 个")

# 5. 创建宏观特征（使用merge_asof对齐时间戳）
print("  [3.5] 创建宏观特征...")
# 创建基准DataFrame - 使用reset_index()确保正确的datetime类型
# 关键：直接赋值 macro_features['_timestamp'] = macro_features.index 会导致类型变为object
temp_macro = main_spread.reset_index()
temp_macro.columns = ['date'] + list(main_spread.columns)

# 确保date列是datetime类型且无时区
temp_macro['date'] = pd.to_datetime(temp_macro['date'],utc=True)
if hasattr(temp_macro['date'].dtype, 'tz') and temp_macro['date'].dtype.tz is not None:
    temp_macro['date'] = temp_macro['date'].dt.tz_localize(None)

print(f"主数据时间类型: {temp_macro['date'].dtype}, 前5行:")
print(temp_macro[['date']].head())

for symbol in ['VIX', 'DXY', 'RB']:
    df = db.get_price_data(symbol)
    if not df.empty and 'close' in df.columns:
        # 准备宏观数据：重置索引，计算收益率
        macro_df = df[['close']].copy()
        macro_df[f'{symbol}_return_1d'] = macro_df['close'].pct_change()
        macro_df = macro_df.reset_index()
        macro_df.columns = ['date', f'{symbol}_close', f'{symbol}_return_1d']
        
        # 确保有干净的datetime列（不带时区）
        # 如果原数据带时区，先用utc=True转换，再移除时区
        macro_df['date'] = pd.to_datetime(macro_df['date'], utc=True)
        if hasattr(macro_df['date'].dtype, 'tz') and macro_df['date'].dtype.tz is not None:
            macro_df['date'] = macro_df['date'].dt.tz_localize(None)
        
        print(f"{symbol}数据时间类型: {macro_df['date'].dtype}")
        
        # 使用merge_asof进行时间对齐（向后填充）
        temp_macro = pd.merge_asof(
            temp_macro.sort_values('date'),
            macro_df[['date', f'{symbol}_close', f'{symbol}_return_1d']].sort_values('date'),
            on='date',
            direction='backward'  # 使用最近的历史数据
        )
        
        # 显示对齐效果
        aligned_count = temp_macro[f'{symbol}_close'].notna().sum()
        missing_count = temp_macro[f'{symbol}_close'].isnull().sum()
        print(f"✓ {symbol}: {len(df)}条原始 → {aligned_count}条对齐（缺失{missing_count}个）")

# 恢复为索引格式（确保索引无时区）
macro_features = temp_macro.set_index('date')
# 再次确认索引无时区
# if hasattr(macro_features.index.dtype, 'tz') and macro_features.index.tz is not None:
#     macro_features.index = macro_features.index.tz_localize(None)

# 只保留宏观数据列，去除来自main_spread的列（避免与spread_features重复）
macro_cols = [col for col in macro_features.columns if any(x in col for x in ['VIX', 'DXY', 'RB'])]
macro_features = macro_features[macro_cols]

print("\n宏观特征前20行:")
print(macro_features.head(20))
print(f"宏观特征索引类型: {macro_features.index.dtype}")

print(f"    ✓ 宏观特征: {len(macro_features.columns)} 个（已时间对齐）")




# ============================================================
# 在特征工程中添加基本面特征
# ============================================================
# 注意：这个单元格应该在原有特征工程单元格中的宏观特征之后添加

# 6. 创建基本面特征（焦炭焦煤仓单和库存数据）
print("  [3.6] 创建基本面特征...")

if 'COAL_INVENTORY' in fundamental_data and not fundamental_data['COAL_INVENTORY'].empty:
    # 获取基本面数据
    coal_inv = fundamental_data['COAL_INVENTORY'].copy()
    
    # 重置索引以便使用merge_asof
    coal_inv_reset = coal_inv.reset_index()
    coal_inv_reset.columns = ['date'] + list(coal_inv.columns)
    
    # 统一时区处理：转换为UTC再移除时区
    coal_inv_reset['date'] = pd.to_datetime(coal_inv_reset['date'], utc=True)
    if hasattr(coal_inv_reset['date'].dtype, 'tz') and coal_inv_reset['date'].dtype.tz is not None:
        coal_inv_reset['date'] = coal_inv_reset['date'].dt.tz_localize(None)
    
    print(f"    基本面数据时间类型: {coal_inv_reset['date'].dtype}")
    
    # 使用主数据的临时DataFrame（temp_macro已经准备好）
    # 使用merge_asof进行时间对齐
    temp_fundamental = temp_macro[['date']].copy()
    
    # 对齐基本面数据
    temp_fundamental = pd.merge_asof(
        temp_fundamental.sort_values('date'),
        coal_inv_reset.sort_values('date'),
        on='date',
        direction='backward'  # 使用最近的历史数据
    )
    
    # 计算基本面数据的变化率
    for col in coal_inv.columns:
        if col in temp_fundamental.columns:
            # 计算变化率
            temp_fundamental[f'{col}_pct_change'] = temp_fundamental[col].pct_change()
            # 计算滚动均值
            temp_fundamental[f'{col}_ma20'] = temp_fundamental[col].rolling(20).mean()
    
    # 恢复为索引格式
    fundamental_features = temp_fundamental.set_index('date')
    
    # 只保留基本面相关列
    fundamental_cols = [col for col in fundamental_features.columns 
                       if any(x in col for x in [
                           'warehouse_receipt', 'inventory', 'inflow',
                           'pct_change', 'ma20'
                       ])]
    fundamental_features = fundamental_features[fundamental_cols]
    
    # 显示对齐效果
    print(f"    ✓ 基本面特征: {len(fundamental_features.columns)} 个")
    aligned_count = fundamental_features.notna().any(axis=1).sum()
    print(f"    ✓ 时间对齐: {aligned_count} 条有效数据")
    
    print("\n    基本面特征列表:")
    for col in fundamental_features.columns:
        non_null = fundamental_features[col].notna().sum()
        print(f"      - {col}: {non_null} 个有效值")
    
else:
    print("    ⚠️  无 COAL_INVENTORY 基本面数据")
    fundamental_features = pd.DataFrame(index=main_spread.index)
    print("    ✓ 创建空的基本面特征DataFrame")



# 6. 创建目标变量
print("  [3.6] 创建目标变量...")
target_df = feature_engineer.create_target_variable(
    main_spread,
    method='sharpe_regress',
    forward_period=10,
)
print(f"    ✓ 目标变量创建完成")

# 🔍 调试点3：在此处设置断点，检查各个特征DataFrame
# 可以查看: spread_features.head(), price_features.shape, technical_features.columns

# ============================================================
# 合并特征
# ============================================================
print("\n  [3.7] 合并所有特征...")
logger.info("合并特征...")

all_features_list = [
    spread_features,
    price_features,
    technical_features,
    fundamental_features,
    macro_features,
    target_df[['target']]
]
# print(pd.concat(all_features_list, axis=1).head(20))

# 诊断：打印合并前的状态
print("\n  诊断信息 - 合并前各DataFrame状态:")

df_names = ['价差特征', '价格特征', '技术指标', '季节性特征', '宏观特征', '目标变量']

# 统一清理所有DataFrame的索引时区
print("\n  [3.7.1] 统一清理索引时区...")
for i, (name, df) in enumerate(zip(df_names, all_features_list)):
    # 检查索引是否有时区
    df.index = pd.to_datetime(df.index, utc=True)
    if hasattr(df.index, 'tz') and df.index.tz is not None:
        print(f"    {name}: 移除时区 {df.index.tz}")
        all_features_list[i].index = df.index.tz_localize(None)
    
    # 打印诊断信息
    null_count = df.isnull().sum().sum()
    index_type = type(df.index).__name__
    index_dtype = df.index.dtype if hasattr(df.index, 'dtype') else 'N/A'
    print(f"    {name}: {len(df)}样本, {len(df.columns)}列, {null_count}缺失值, 索引类型:{index_type}({index_dtype})")
    if null_count > 0:
        null_cols = df.isnull().sum()
        null_cols = null_cols[null_cols > 0]
        print(f"      缺失值列: {dict(list(null_cols.items())[:3])}")

# 合并特征
print("\n  [3.7.2] 开始合并特征...")
features_df = feature_engineer.merge_all_features(all_features_list)

print(f"\n  合并后状态:")
print(f"    总样本数: {len(features_df)}")
print(f"    总特征数: {len(features_df.columns)}")
print(f"    总缺失值: {features_df.isnull().sum().sum()}")

# ============================================================
# 清理缺失值
# ============================================================
print("\n  [3.8] 清理缺失值...")

# 显示缺失值最多的列
total_nulls = features_df.isnull().sum().sum()
if total_nulls > 0:
    null_counts = features_df.isnull().sum()
    cols_with_nulls = null_counts[null_counts > 0].sort_values(ascending=False)
    print(f"    缺失值最多的前5列:")
    for col, count in cols_with_nulls.head(5).items():
        pct = count / len(features_df) * 100
        print(f"      {col}: {count} ({pct:.2f}%)")

# 步骤1：前向填充
print("\n    步骤1: ffill前向填充...")
exclude_target = features_df.columns.difference(['target', 'forward_return'])
features_df[exclude_target] = features_df[exclude_target].fillna(method='ffill')
remaining_nulls = features_df.isnull().sum().sum()
print(f"      剩余缺失值: {remaining_nulls}")

# # 步骤2：后向填充
# if remaining_nulls > 0:
#     print("    步骤2: bfill后向填充...")
#     features_df = features_df.fillna(method='bfill')
#     remaining_nulls = features_df.isnull().sum().sum()
#     print(f"      剩余缺失值: {remaining_nulls}")

# # 步骤3：均值填充
# if remaining_nulls > 0:
#     print("    步骤3: 均值填充...")
#     numeric_cols = features_df.select_dtypes(include=[np.number]).columns
#     features_df[numeric_cols] = features_df[numeric_cols].fillna(
#         features_df[numeric_cols].mean()
#     )
#     remaining_nulls = features_df.isnull().sum().sum()
#     print(f"      剩余缺失值: {remaining_nulls}")

# 步骤4：删除仍有缺失值的列
# if remaining_nulls > 0:
#     null_cols = features_df.columns[features_df.isnull().any()].tolist()
#     print(f"    步骤4: 删除 {len(null_cols)} 个仍有缺失值的列")
#     print(f"      删除的列: {null_cols[:5]}")
#     features_df = features_df.dropna(axis=1)

print(f"\n  最终清理结果:")
print(f"    剩余样本数: {len(features_df)}")
print(f"    剩余特征数: {len(features_df.columns) - 2}")  # 减去target和forward_return
print(f"    缺失值: {features_df.isnull().sum().sum()}")

print("✓ 特征工程完成")
logger.info(f"特征构建完成，总特征数: {len(features_df.columns) - 2}")
logger.info(f"样本数: {len(features_df)}")

# 🔍 调试点4：在此处设置断点，检查 features_df
# 可以使用: features_df.describe(), features_df.head(), features_df.isnull().sum()

INFO:__main__:
INFO:__main__:步骤3：特征工程
INFO:__main__:============================================================
INFO:src.core.feature_engineering:创建价差特征完成，特征数: 39
INFO:src.core.feature_engineering:创建价格特征完成，特征数: 32



[4/9] 开始特征工程...
  [3.1] 创建价差特征...
    ✓ 价差特征: 51 个
  [3.2] 创建价格特征...
    ✓ 价格特征: 32 个
  [3.3] 创建技术指标特征...
    ✓ 技术指标特征: 0 个
  [3.5] 创建宏观特征...
主数据时间类型: datetime64[ns], 前5行:
                 date
0 2013-03-21 16:00:00
1 2013-03-24 16:00:00
2 2013-03-25 16:00:00
3 2013-03-26 16:00:00
4 2013-03-27 16:00:00
VIX数据时间类型: datetime64[ns]
✓ VIX: 6498条原始 → 3059条对齐（缺失0个）


INFO:src.core.feature_engineering:创建夏普比率回归目标变量
INFO:__main__:合并特征...
INFO:src.core.feature_engineering:特征合并完成，总特征数: 102, 样本数: 3059


DXY数据时间类型: datetime64[ns]
✓ DXY: 6528条原始 → 3059条对齐（缺失0个）
RB数据时间类型: datetime64[ns]
✓ RB: 4031条原始 → 3059条对齐（缺失0个）

宏观特征前20行:
                     VIX_close  VIX_return_1d  DXY_close  DXY_return_1d  \
date                                                                      
2013-03-21 16:00:00  13.990000       0.104183  82.800003       0.000242   
2013-03-24 16:00:00  13.570000      -0.030021  82.529999      -0.003261   
2013-03-25 16:00:00  13.740000       0.012528  82.870003       0.004120   
2013-03-26 16:00:00  12.770000      -0.070597  82.879997       0.000121   
2013-03-27 16:00:00  13.150000       0.029757  83.209999       0.003982   
2013-03-28 16:00:00  12.700000      -0.034221  82.989998      -0.002644   
2013-03-31 16:00:00  12.700000      -0.034221  82.989998      -0.002644   
2013-04-01 16:00:00  13.580000       0.069291  82.730003      -0.003133   
2013-04-02 16:00:00  12.780000      -0.058910  82.940002       0.002538   
2013-04-07 16:00:00  13.920000       0.002160  82.50

INFO:__main__:特征构建完成，总特征数: 100
INFO:__main__:样本数: 3059


      剩余缺失值: 1667

  最终清理结果:
    剩余样本数: 3059
    剩余特征数: 100
    缺失值: 1667
✓ 特征工程完成


In [15]:
# ============================================================
# 测试：查看基本面数据和特征
# ============================================================
print("📊 查看焦炭焦煤基本面数据")
print("="*60)

if 'COAL_INVENTORY' in fundamental_data and not fundamental_data['COAL_INVENTORY'].empty:
    coal_inv = fundamental_data['COAL_INVENTORY']
    
    print(f"\n数据基本信息:")
    print(f"  样本数: {len(coal_inv)}")
    print(f"  时间范围: {coal_inv.index.min()} 至 {coal_inv.index.max()}")
    print(f"  时区: {coal_inv.index.tz}")
    
    print(f"\n数据字段:")
    for col in coal_inv.columns:
        print(f"  - {col}")
    
    print(f"\n数据统计:")
    print(coal_inv.describe())
    
    print(f"\n最新数据（最近5条）:")
    print(coal_inv.tail(5))
    
    print(f"\n数据缺失情况:")
    missing = coal_inv.isnull().sum()
    for col, count in missing.items():
        pct = count / len(coal_inv) * 100
        print(f"  {col}: {count} ({pct:.1f}%)")
    
else:
    print("⚠️  未找到 COAL_INVENTORY 数据")
    print("请先运行数据导入单元格")

print("\n" + "="*60)

📊 查看焦炭焦煤基本面数据

数据基本信息:
  样本数: 6659
  时间范围: 2002-01-23 00:00:00+08:00 至 2025-10-31 00:00:00+08:00
  时区: Asia/Shanghai

数据字段:
  - bohai_port_inventory
  - bohai_port_rail_inflow
  - coke_warehouse_receipt
  - coking_coal_warehouse_receipt

数据统计:
       bohai_port_inventory  bohai_port_rail_inflow  coke_warehouse_receipt  \
count           6641.000000             6245.000000             5183.000000   
mean            1273.118192              155.654296              212.126182   
std              581.497336               55.624995              431.376534   
min                0.000000               21.100000                0.000000   
25%              730.600000              112.500000                0.000000   
50%             1262.700000              159.100000                0.000000   
75%             1781.220000              185.400000              160.000000   
max             2694.700000              388.400000             6530.000000   

       coking_coal_warehouse_receipt  
count

In [16]:
features_df = features_df.dropna()

In [17]:
import importlib
import src.core.ml_models
importlib.reload(src.core.ml_models)
from src.core.ml_models import MLModel,SignalGenerator

In [18]:
# ============================================================
# 步骤4：模型训练
# ============================================================
print("\n[5/9] 开始模型训练...")
logger.info("\n" + "="*60)
logger.info("步骤4：模型训练")
logger.info("="*60)

if features_df is None or features_df.empty:
    print("  ✗ 特征数据不可用，停止执行")
    logger.error("特征数据不可用")
    raise ValueError("特征数据不可用")

# 创建模型
print("  [4.1] 创建模型...")
model = MLModel(model_type='gradient_boosting', task='regression')
print("    ✓ 使用 Gradient Boosting 分类器")

# 准备数据 - 先划分训练集和测试集（避免数据泄露）
print("  [4.2] 准备训练/测试数据...")
X_train, X_test, y_train, y_test, train_idx, test_idx = model.prepare_data(
    features_df,
    target_col='target',
    feature_cols=None,  # 暂时使用所有特征
    test_size=0.2,
    scale=False  # 暂时不标准化，特征选择后再标准化
)
print(f"    ✓ 训练集: {len(X_train)} 样本")
print(f"    ✓ 测试集: {len(X_test)} 样本")

# 特征选择 - 仅在训练集上进行（避免数据泄露）
print("  [4.3] 特征选择（仅基于训练集）...")
# 重建训练集DataFrame用于特征选择
all_feature_cols = [col for col in features_df.columns if col != 'target']
train_df = pd.DataFrame(X_train, columns=all_feature_cols, index=train_idx)
train_df['target'] = y_train

selected_features = feature_engineer.select_features(
    train_df,
    target_col='target',
    method='variance',
    top_k=50
)
print(f"    ✓ 选择了 {len(selected_features)} 个特征")

# 使用选定的特征重新准备数据
print("  [4.4] 使用选定特征重新准备数据...")
X_train_selected, X_test_selected, y_train, y_test, train_idx, test_idx = model.prepare_data(
    features_df,
    target_col='target',
    feature_cols=selected_features,
    test_size=0.2,
    scale=True  # 现在进行标准化
)
print(f"    ✓ 训练集: {len(X_train_selected)} 样本, {X_train_selected.shape[1]} 个特征")
print(f"    ✓ 测试集: {len(X_test_selected)} 样本, {X_test_selected.shape[1]} 个特征")

# 训练模型
print("  [4.5] 训练模型...")
model_params = {
    'n_estimators': 500,
    'max_depth': 8,
    'learning_rate': 0.1,
    'random_state': 42
}
model.train(X_train_selected, y_train, **model_params)
print("    ✓ 模型训练完成")

# 评估模型
print("  [4.6] 评估模型...")
metrics = model.evaluate(X_test_selected, y_test)
print(f"    ✓ 准确率: {metrics.get('accuracy', 0):.4f}")
print(f"    ✓ F1分数: {metrics.get('f1', 0):.4f}")

# 可视化特征重要性
if model.feature_importance is not None:
    print("  [4.7] 生成特征重要性图...")
    visualizer.plot_feature_importance(model.feature_importance, top_n=15)
    print("    ✓ 特征重要性图已保存")

# 保存模型
print("  [4.8] 保存模型...")
model_path = Path('models/coking_model.pkl')
model_path.parent.mkdir(exist_ok=True)
model.save_model(str(model_path))
print(f"    ✓ 模型已保存到: {model_path}")

print("✓ 模型训练完成")
logger.info("模型训练完成\n")

# 🔍 调试点5：在此处设置断点，检查模型和训练结果
# 可以查看: model.feature_importance, metrics, X_train_selected.shape, X_test_selected.shape

# ============================================================
# 步骤5：回测
# ============================================================
print("\n[6/9] 开始回测...")
logger.info("\n" + "="*60)
logger.info("步骤5：回测")
logger.info("="*60)


INFO:__main__:
INFO:__main__:步骤4：模型训练
INFO:__main__:============================================================
INFO:src.core.ml_models:数据准备完成:
INFO:src.core.ml_models:  特征数: 101
INFO:src.core.ml_models:  训练集样本数: 2331
INFO:src.core.ml_models:  测试集样本数: 583



[5/9] 开始模型训练...
  [4.1] 创建模型...
    ✓ 使用 Gradient Boosting 分类器
  [4.2] 准备训练/测试数据...
    ✓ 训练集: 2331 样本
    ✓ 测试集: 583 样本
  [4.3] 特征选择（仅基于训练集）...


INFO:src.core.feature_engineering:特征选择完成，选择了 50 个特征
INFO:src.core.ml_models:数据准备完成:
INFO:src.core.ml_models:  特征数: 50
INFO:src.core.ml_models:  训练集样本数: 2331
INFO:src.core.ml_models:  测试集样本数: 583
INFO:src.core.ml_models:开始训练 gradient_boosting 模型...


    ✓ 选择了 50 个特征
  [4.4] 使用选定特征重新准备数据...
    ✓ 训练集: 2331 样本, 50 个特征
    ✓ 测试集: 583 样本, 50 个特征
  [4.5] 训练模型...


INFO:src.core.ml_models:Top 10 重要特征:
INFO:src.core.ml_models:                      feature  importance
40              spread_std_60    0.073480
33               spread_ma_60    0.065211
14  bohai_port_inventory_ma20    0.055119
46                spread_macd    0.040503
5                     J_ma_20    0.040006
38           distance_to_high    0.037077
6                     J_ma_50    0.035959
0              J_volume_ma_20    0.032783
21                     spread    0.031441
1             JM_volume_ma_20    0.029315
INFO:src.core.ml_models:模型训练完成
INFO:src.core.ml_models:
模型评估结果:
INFO:src.core.ml_models:MSE: 5.491222
INFO:src.core.ml_models:RMSE: 2.343336
INFO:src.core.ml_models:MAE: 1.878188
INFO:src.core.ml_models:R²: -0.2415


    ✓ 模型训练完成
  [4.6] 评估模型...
    ✓ 准确率: 0.0000
    ✓ F1分数: 0.0000
  [4.7] 生成特征重要性图...


INFO:src.core.visualization:特征重要性图表已保存: outputs/Coking_Coal&Coke_Results/feature_importance.png
INFO:src.core.ml_models:模型已保存: models\coking_model.pkl
INFO:__main__:模型训练完成

INFO:__main__:
INFO:__main__:步骤5：回测
INFO:__main__:============================================================


    ✓ 特征重要性图已保存
  [4.8] 保存模型...
    ✓ 模型已保存到: models\coking_model.pkl
✓ 模型训练完成

[6/9] 开始回测...


In [20]:
model.predict(X_test_selected)

array([-1.76768426e+00,  2.31418098e-01,  2.86212613e-01, -1.70203153e-01,
        9.97586502e-01,  2.28195683e-01,  1.45754652e-01, -1.01867608e-01,
       -1.55757606e-01, -1.90888366e-01, -5.01425113e-01, -5.62106046e-01,
       -1.26158021e+00, -1.54612088e+00, -1.31067382e+00, -1.73887037e+00,
       -1.72290387e+00, -4.41034479e-01, -1.37204151e+00, -1.53115542e+00,
       -1.40850782e+00, -1.48178068e+00, -1.15278279e+00, -6.94247028e-01,
       -6.32968531e-01, -7.32381845e-01,  2.16178151e-01,  3.36229189e-01,
       -1.07879266e-01,  4.32304086e-01,  5.61596996e-01,  3.17084561e-01,
       -6.02679458e-02,  2.42286542e-02, -3.24204283e-01, -8.52843295e-01,
       -9.91005419e-01, -1.04945544e+00, -4.88368734e-01, -8.34466639e-01,
       -2.13636216e-01, -4.75596362e-01, -8.58492720e-01, -9.33883592e-01,
       -4.75408837e-01, -5.75598544e-01, -6.16632201e-01, -1.43970802e+00,
       -2.25810055e+00, -7.39075501e-01,  1.30840774e-01, -3.11256531e-01,
       -3.69433060e-01, -

In [ ]:
# 生成信号
print("  [5.1] 生成交易信号...")
signal_generator = SignalGenerator(model, 
                                use_rolling_quantile=True,      # ✅ 使用滚动分位数
                                rolling_window=250,              # 250天窗口
                                upper_quantile=0.8,            # 80%分位数
                                lower_quantile=0.2,            # 20%分位数
                                signal_holding_days=10          # 信号维持10天
                                   )
signals = signal_generator.generate_signals(X_test_selected, use_probability=True)
signals.index = test_idx
print(f"    ✓ 生成 {len(signals)} 个信号")
print(f"    信号分布: {signals.value_counts().to_dict()}")

# 获取价差价格数据
print("  [5.2] 准备价格数据...")
# 确保spread_data索引与test_idx时区一致
spread_df_for_backtest = spread_data['COKING_PROFIT'].copy()
spread_df_for_backtest.index = pd.to_datetime(spread_df_for_backtest.index,utc=True)
if hasattr(spread_df_for_backtest.index, 'tz') and spread_df_for_backtest.index.tz is not None:
    spread_df_for_backtest.index = spread_df_for_backtest.index.tz_localize(None)

spread_prices = spread_df_for_backtest.loc[test_idx, ['spread']].copy()
spread_prices.columns = ['close']
spread_prices['volatility'] = spread_prices['close'].pct_change().rolling(20).std()
print(f"    ✓ 价格数据: {len(spread_prices)} 条")


# 运行回测
print("  [5.3] 运行回测...")
backtest_engine = BacktestEngine(
    initial_capital=1000000,
    commission_rate=0.0005,
    slippage_rate=0.0001,
    max_position=100000,
    max_capital_usage=0.3,
    leverage=1,
    stop_loss_pct=0.0
)

equity_curve = backtest_engine.run_backtest(
    spread_prices,
    signals,
    price_col='close',
    volatility_col='volatility'
)
print(f"    ✓ 回测完成，最终权益: ${equity_curve['equity'].iloc[-1]:,.2f}")

# 获取交易日志
trade_log = backtest_engine.get_trade_log()
print(f"    ✓ 总交易次数: {len(trade_log)}")

# 绩效分析
print("  [5.4] 绩效分析...")
analyzer = PerformanceAnalyzer(
    equity_curve,
    initial_capital=1000000,
    risk_free_rate=0.02
)

performance_report = analyzer.generate_performance_report(trade_log)
print(f"    ✓ 总收益率: {performance_report.get('total_return', 0)*100:.2f}%")
print(f"    ✓ 夏普比率: {performance_report.get('sharpe_ratio', 0):.2f}")
print(f"    ✓ 最大回撤: {performance_report.get('max_drawdown', 0)*100:.2f}%")

print("✓ 回测完成")
logger.info("回测完成\n")

# 🔍 调试点6：在此处设置断点，检查回测结果
# 可以查看: equity_curve.tail(), trade_log.head(), performance_report

# ============================================================
# 步骤6：可视化
# ============================================================
print("\n[7/9] 开始可视化...")
logger.info("\n" + "="*60)
logger.info("步骤6：结果可视化")
logger.info("="*60)

print("  [6.1] 生成价格和价差图...")
visualizer.plot_price_and_spread(
    price_data,
    spread_data['COKING_PROFIT'],
    title='COKING_PROFIT 10:13'
)
print("    ✓ price_spread_chart.png")

print("  [6.2] 生成权益曲线图...")
visualizer.plot_equity_curve(equity_curve)
print("    ✓ equity_curve.png")

print("  [6.3] 生成收益率分布图...")
returns = equity_curve['equity'].pct_change().dropna()
visualizer.plot_returns_distribution(returns)
print("    ✓ returns_distribution.png")

print("  [6.4] 生成月度收益热力图...")
visualizer.plot_monthly_returns_heatmap(equity_curve)
print("    ✓ monthly_returns_heatmap.png")

print("  [6.5] 生成滚动指标图...")
visualizer.plot_rolling_metrics(equity_curve, window=60)
print("    ✓ rolling_metrics.png")

print("  [6.6] 生成交易分析图...")
visualizer.plot_trade_analysis(trade_log)
print("    ✓ trade_analysis.png")

print("✓ 可视化完成")
logger.info("可视化完成\n")

# ============================================================
# 完成
# ============================================================
print("\n" + "="*60)
print("策略执行完成！")
print("="*60)
print("\n生成的文件:")
print("  📁 data/trading_data.db          - 数据库")
print("  📁 models/Coking_Profit_model.pkl - 模型文件")
print("  📁 outputs/charts/*.png          - 图表文件")
print("  📁 logs/debug_strategy.log       - 日志文件")

print("\n可用的全局变量（用于调试）:")
print("  数据相关:")
print("    - price_data       : 期货价格数据字典")
print("    - spread_data      : 价差数据字典")
print("    - macro_data       : 宏观数据字典")
print("    - fundamental_data : 基本面数据字典")
print("\n  特征相关:")
print("    - spread_features  : 价差特征DataFrame")
print("    - price_features   : 价格特征DataFrame")
print("    - technical_features: 技术指标特征DataFrame")
print("    - seasonal_features: 季节性特征DataFrame")
print("    - macro_features   : 宏观特征DataFrame")
print("    - target_df        : 目标变量DataFrame")
print("    - features_df      : 合并后的完整特征DataFrame")
print("\n  模型相关:")
print("    - model            : 训练好的模型")
print("    - X_train, X_test  : 训练/测试特征")
print("    - y_train, y_test  : 训练/测试标签")
print("    - selected_features: 选择的特征列表")
print("\n  回测相关:")
print("    - signals          : 交易信号Series")
print("    - equity_curve     : 权益曲线DataFrame")
print("    - trade_log        : 交易日志DataFrame")
print("    - performance_report: 绩效报告字典")

print("\n💡 调试提示:")
print("  1. 在VS Code中打开此文件")
print("  2. 点击行号左侧设置断点（蓝点）")
print("  3. 按F5或点击'运行和调试'启动调试")
print("  4. 在'变量'面板查看所有变量的值")
print("  5. 在'调试控制台'输入变量名查看详细信息")
print("  例如: price_data.keys(), features_df.shape, model.feature_importance")

logger.info("\n" + "="*60)
logger.info("所有任务完成！")
logger.info("="*60)

# 🔍 最终调试点：程序结束前，所有变量都已计算完成
# 现在可以检查任何变量的最终状态

INFO:src.core.ml_models:使用滚动分位数模式: window=250, 上分位数=0.8, 下分位数=0.2
INFO:src.core.ml_models:滚动分位数统计:
INFO:src.core.ml_models:  上阈值范围: [0.2999, 1.8958], 均值: 1.0221
INFO:src.core.ml_models:  下阈值范围: [-1.3843, -0.4811], 均值: -1.1509
INFO:src.core.ml_models:回归信号统计:
INFO:src.core.ml_models:  预测值范围: [-2.2581, 3.9227]
INFO:src.core.ml_models:  做多信号(1): 133 (22.8%)
INFO:src.core.ml_models:  观望信号(0): 361 (61.9%)
INFO:src.core.ml_models:  做空信号(-1): 89 (15.3%)
INFO:src.core.ml_models:生成交易信号完成，信号分布:
INFO:src.core.ml_models: 0    361
 1    133
-1     89
Name: signal, dtype: int64
INFO:src.core.ml_models:应用10天信号维持后，信号分布:
INFO:src.core.ml_models: 1    256
-1    231
 0     96
Name: signal, dtype: int64


  [5.1] 生成交易信号...
    ✓ 生成 583 个信号
    信号分布: {(-2.2581005488953934, 0.3033055402106255, -1.3842556693105381, -1, 0.5178152203633574): 1, (0.4352532631948662, 1.8957721107803633, -0.8841821911179734, 0, 0.0): 1, (0.6061424234264167, 0.4502868194954966, -1.272057773697288, 1, 0.09049037256940773): 1, (0.6067720325233416, 1.3644844334390238, -1.1230816389029104, 1, 0.0): 1, (0.607319170224082, 1.480116811504026, -0.4810671720919284, -1, 0.0): 1, (0.6395944271062602, 1.703697663312618, -0.737795304789794, 1, 0.0): 1, (0.6439708025058702, 1.3500419568715423, -1.1462151061779782, 0, 0.0): 1, (0.658583564475944, 1.8957721107803633, -0.7517556307474458, -1, 0.0): 1, (0.6626638285450346, 0.3033055402106255, -1.3842556693105381, 1, 0.2129453357347406): 1, (0.6733673106666375, 0.3033055402106255, -1.3842556693105381, 1, 0.21928790989514094): 1, (0.6815444913987198, 0.3033055402106255, -1.3842556693105381, 1, 0.22413347086557975): 1, (0.6942825122190048, 0.31363980605572006, -1.3811236634190678, 1

INFO:src.core.backtest:开始运行回测...
INFO:src.core.backtest:初始资金: $1,000,000.00
INFO:src.core.backtest:手续费率: 0.050%
INFO:src.core.backtest:滑点率: 0.010%
INFO:src.core.backtest:杠杆倍数: 1x
INFO:src.core.backtest:保证金比例: 10.0%
INFO:src.core.backtest:最大资金使用率: 30%
INFO:src.core.backtest:回测完成，共执行 473 笔交易
INFO:src.core.backtest:最终权益: $2,544,901.98
INFO:src.core.backtest:
INFO:src.core.backtest:绩效分析报告
INFO:src.core.backtest:============================================================
INFO:src.core.backtest:总收益率: 154.49%
INFO:src.core.backtest:年化收益率: 49.39%
INFO:src.core.backtest:年化波动率: 38.12%
INFO:src.core.backtest:夏普比率: 1.2430
INFO:src.core.backtest:索提诺比率: 1.5586
INFO:src.core.backtest:卡玛比率: 1.8692
INFO:src.core.backtest:最大回撤: -26.42%
INFO:src.core.backtest:VaR (95%): -1.92%
INFO:src.core.backtest:CVaR (95%): -4.27%
INFO:src.core.backtest:胜率: 55.76%
INFO:src.core.backtest:盈亏比: 2.60
INFO:src.core.backtest:总交易次数: 473
INFO:src.core.backtest:总手续费: $9,349.23
INFO:src.core.backtest:=========================

    ✓ 价格数据: 583 条
  [5.3] 运行回测...
    ✓ 回测完成，最终权益: $2,544,901.98
    ✓ 总交易次数: 473
  [5.4] 绩效分析...
    ✓ 总收益率: 154.49%
    ✓ 夏普比率: 1.24
    ✓ 最大回撤: -26.42%
✓ 回测完成

[7/9] 开始可视化...
  [6.1] 生成价格和价差图...


INFO:src.core.visualization:价格和价差图表已保存: outputs/Coking_Coal&Coke_Results/price_spread_chart.png


...

    ✓ monthly_returns_heatmap.png
  [6.5] 生成滚动指标图...


INFO:src.core.visualization:滚动指标图表已保存: outputs/Coking_Coal&Coke_Results/rolling_metrics.png


    ✓ rolling_metrics.png
  [6.6] 生成交易分析图...


INFO:src.core.visualization:交易分析图表已保存: outputs/Coking_Coal&Coke_Results/trade_analysis.png
INFO:__main__:可视化完成

INFO:__main__:
INFO:__main__:所有任务完成！
INFO:__main__:============================================================


    ✓ trade_analysis.png
✓ 可视化完成

策略执行完成！

生成的文件:
  📁 data/trading_data.db          - 数据库
  📁 models/Coking_Profit_model.pkl - 模型文件
  📁 outputs/charts/*.png          - 图表文件
  📁 logs/debug_strategy.log       - 日志文件

可用的全局变量（用于调试）:
  数据相关:
    - price_data       : 期货价格数据字典
    - spread_data      : 价差数据字典
    - macro_data       : 宏观数据字典
    - fundamental_data : 基本面数据字典

  特征相关:
    - spread_features  : 价差特征DataFrame
    - price_features   : 价格特征DataFrame
    - technical_features: 技术指标特征DataFrame
    - seasonal_features: 季节性特征DataFrame
    - macro_features   : 宏观特征DataFrame
    - target_df        : 目标变量DataFrame
    - features_df      : 合并后的完整特征DataFrame

  模型相关:
    - model            : 训练好的模型
    - X_train, X_test  : 训练/测试特征
    - y_train, y_test  : 训练/测试标签
    - selected_features: 选择的特征列表

  回测相关:
    - signals          : 交易信号Series
    - equity_curve     : 权益曲线DataFrame
    - trade_log        : 交易日志DataFrame
    - performance_report: 绩效报告字典

💡 调试提示:
  1. 在VS Code中打开此文件
  2. 点击行号左侧设置断点（

In [24]:
# ============================================================
# 步骤7：保存结果
# ============================================================
print("\n[8/9] 保存策略结果...")
logger.info("\n" + "="*60)
logger.info("步骤7：保存结果")
logger.info("="*60)

# 重新导入模块以获取最新代码
import importlib
import src.core.result_saver
importlib.reload(src.core.result_saver)
from src.core.result_saver import ResultSaver
from datetime import datetime
from pathlib import Path

# 创建结果保存器
result_saver = ResultSaver(output_dir="outputs/Coking_Coal&Coke_Results/strategy_runs")

# 1. 保存策略配置
print("  [7.1] 保存策略配置...")

# 提取模型参数
model_params = {
    "model_type": "gradient_boosting",
    "task": model.task,
    "n_estimators": model.model.n_estimators if hasattr(model.model, 'n_estimators') else None,
    "max_depth": model.model.max_depth if hasattr(model.model, 'max_depth') else None,
    "learning_rate": model.model.learning_rate if hasattr(model.model, 'learning_rate') else None,
    "random_state": 42,
    "scaler_used": model.scaler is not None,
    "feature_count": len(selected_features)
}

# 提取信号生成参数
signal_params = {
    "use_rolling_quantile": signal_generator.use_rolling_quantile,
    "rolling_window": signal_generator.rolling_window,
    "upper_quantile": signal_generator.upper_quantile,
    "lower_quantile": signal_generator.lower_quantile,
    "signal_holding_days": signal_generator.signal_holding_days,
    "use_probability": True
}

# 提取回测参数
backtest_params = {
    "initial_capital": backtest_engine.initial_capital,
    "commission_rate": backtest_engine.commission_rate,
    "slippage_rate": backtest_engine.slippage_rate,
    "max_position": backtest_engine.max_position,
    "max_capital_usage": backtest_engine.max_capital_usage
}

# 数据信息
data_info = {
    "start_date": START_DATE,
    "end_date": datetime.now().strftime('%Y-%m-%d'),
    "symbols": list(price_data.keys()),
    "spread_type": "Coking_Profit",
    "train_samples": len(X_train),
    "test_samples": len(X_test),
    "train_period": f"{train_idx.min()} to {train_idx.max()}",
    "test_period": f"{test_idx.min()} to {test_idx.max()}",
    "total_features": len(selected_features),
    "feature_categories": {
        "spread_features": len([f for f in selected_features if 'spread' in f.lower()]),
        "price_features": len([f for f in selected_features if any(s in f for s in ['JM_', 'J_',])]),
        "technical_features": len([f for f in selected_features if any(s in f for s in ['RSI', 'MACD', 'BB'])]),
        "macro_features": len([f for f in selected_features if any(s in f for s in ['VIX', 'DXY', 'RB'])]),
        "fundamental_features": len([f for f in selected_features if any(s in f for s in ['warehouse_receipt', 'inventory', 'inflow'])]),
    }
}

config_file = result_saver.save_strategy_config(
    model_params=model_params,
    signal_params=signal_params,
    backtest_params=backtest_params,
    data_info=data_info
)
print(f"    ✓ 策略配置已保存")

# 2. 保存绩效报告
print("  [7.2] 保存绩效报告...")
perf_files = result_saver.save_performance_report(
    performance_report=performance_report,
    trade_log=trade_log,
    equity_curve=equity_curve
)
print(f"    ✓ 绩效报告已保存:")
for file_type, filepath in perf_files.items():
    print(f"      - {file_type}: {Path(filepath).name}")

# 3. 保存特征信息
print("  [7.3] 保存特征信息...")
feature_file = result_saver.save_feature_info(
    selected_features=selected_features,
    feature_importance=model.feature_importance
)
print(f"    ✓ 特征信息已保存")

# 4. 保存模型指标
print("  [7.4] 保存模型指标...")
train_metrics = model.evaluate(X_train_selected, y_train)
test_metrics = metrics  # 使用之前计算的测试集指标
metrics_file = result_saver.save_model_metrics(
    train_metrics=train_metrics,
    test_metrics=test_metrics
)
print(f"    ✓ 模型指标已保存")
print(f"      训练集准确率: {train_metrics.get('accuracy', 0):.4f}")
print(f"      测试集准确率: {test_metrics.get('accuracy', 0):.4f}")

# 5. 保存交易信号
print("  [7.5] 保存交易信号...")
# 获取概率（如果有）
try:
    if hasattr(model.model, 'predict_proba'):
        probabilities = pd.DataFrame(
            model.model.predict_proba(X_test),
            index=test_idx,
            columns=[f'prob_class_{i}' for i in range(len(model.model.classes_))]
        )
    else:
        probabilities = None
except Exception as e:
    logger.warning(f"无法获取预测概率: {e}")
    probabilities = None

signals_file = result_saver.save_signals(
    signals=signals,
    probabilities=probabilities
)
print(f"    ✓ 交易信号已保存")

# 6. 创建README
print("  [7.6] 创建README...")
result_saver.create_readme()
print(f"    ✓ README已创建")

# 显示输出目录
output_dir = result_saver.get_run_directory()
print(f"\n✓ 所有结果已保存到: {output_dir}")
logger.info(f"所有结果已保存到: {output_dir}\n")

# 打印目录结构
print("\n📂 生成的文件:")
for file in sorted(Path(output_dir).glob('*')):
    size_kb = file.stat().st_size / 1024
    print(f"  📄 {file.name:<35} ({size_kb:>8.2f} KB)")

print("\n" + "="*80)
print("💾 结果保存完成！")
print("="*80)

print("\n📊 快速查看:")
print(f"  绩效汇总: {output_dir}/performance_summary.txt")
print(f"  交易记录: {output_dir}/performance_trades.csv")
print(f"  策略配置: {output_dir}/strategy_config.json")

print("\n💡 提示:")
print("  - 所有CSV文件可以用Excel直接打开")
print("  - JSON文件包含完整的配置和指标信息")
print("  - 不同运行的结果通过时间戳文件夹区分")
print("  - 可以对比不同参数设置的效果")


INFO:__main__:
INFO:__main__:步骤7：保存结果
INFO:__main__:============================================================
INFO:src.core.result_saver:结果保存器初始化完成，输出目录: outputs\Coking_Coal&Coke_Results\strategy_runs\20251103_164830
INFO:src.core.result_saver:策略配置已保存: outputs\Coking_Coal&Coke_Results\strategy_runs\20251103_164830\strategy_config.json
INFO:src.core.result_saver:绩效报告JSON已保存: outputs\Coking_Coal&Coke_Results\strategy_runs\20251103_164830\performance_report.json
INFO:src.core.result_saver:绩效报告CSV已保存: outputs\Coking_Coal&Coke_Results\strategy_runs\20251103_164830\performance_report.csv


INFO:src.core.result_saver:交易日志已保存: outputs\Coking_Coal&Coke_Results\strategy_runs\20251103_164830\performance_trades.csv
INFO:src.core.result_saver:权益曲线已保存: outputs\Coking_Coal&Coke_Results\strategy_runs\20251103_164830\performance_equity_curve.csv
INFO:src.core.result_saver:文本汇总报告已保存: outputs\Coking_Coal&Coke_Results\strategy_runs\20251103_164830\performance_summary.txt
INFO:src.core.result_saver:特征重要性CSV已保存: outputs\Coking_Coal&Coke_Results\strategy_runs\20251103_164830\feature_importance.csv
INFO:src.core.result_saver:特征信息已保存: outputs\Coking_Coal&Coke_Results\strategy_runs\20251103_164830\feature_info.json
INFO:src.core.ml_models:
模型评估结果:
INFO:src.core.ml_models:MSE: 0.000100
INFO:src.core.ml_models:RMSE: 0.010017
INFO:src.core.ml_models:MAE: 0.007886
INFO:src.core.ml_models:R²: 1.0000
INFO:src.core.result_saver:模型指标已保存: outputs\Coking_Coal&Coke_Results\strategy_runs\20251103_164830\model_metrics.json
INFO:src.core.result_saver:交易信号已保存: outputs\Coking_Coal&Coke_Results\strategy_run


[8/9] 保存策略结果...
  [7.1] 保存策略配置...
    ✓ 策略配置已保存
  [7.2] 保存绩效报告...
    ✓ 绩效报告已保存:
      - report_json: performance_report.json
      - report_csv: performance_report.csv
      - trades: performance_trades.csv
      - equity_curve: performance_equity_curve.csv
      - summary_txt: performance_summary.txt
  [7.3] 保存特征信息...
    ✓ 特征信息已保存
  [7.4] 保存模型指标...
    ✓ 模型指标已保存
      训练集准确率: 0.0000
      测试集准确率: 0.0000
  [7.5] 保存交易信号...
    ✓ 交易信号已保存
  [7.6] 创建README...
    ✓ README已创建

✓ 所有结果已保存到: outputs\Coking_Coal&Coke_Results\strategy_runs\20251103_164830

📂 生成的文件:
  📄 feature_importance.csv              (    1.81 KB)
  📄 feature_info.json                   (    6.90 KB)
  📄 model_metrics.json                  (    0.39 KB)
  📄 performance_equity_curve.csv        (   71.08 KB)
  📄 performance_report.csv              (    0.41 KB)
  📄 performance_report.json             (    0.50 KB)
  📄 performance_summary.txt             (    1.57 KB)
  📄 performance_trades.csv              (   37.62 KB)
  📄

# 超参数优化

使用不同的搜索方法优化模型超参数：
- 网格搜索（Grid Search）：遍历所有参数组合
- 随机搜索（Random Search）：随机采样参数组合
- 贝叶斯优化（Bayesian Optimization）：智能搜索最优参数

## 步骤：
1. 导入超参数优化模块
2. 选择搜索方法
3. 执行参数搜索
4. 比较不同方法的结果
5. 使用最佳参数重新训练模型

In [12]:
# 导入超参数优化模块
from src.core.hyperparameter_tuning import HyperparameterTuner

# 创建超参数优化器
tuner = HyperparameterTuner(
    model_type='gradient_boosting',
    task='classification',
    scoring='f1_weighted',  # 使用加权F1分数
    cv=5,  # 5折交叉验证
    n_jobs=-1,  # 使用所有CPU核心
    verbose=1
)

print("超参数优化器初始化完成")
print(f"模型类型: {tuner.model_type}")
print(f"评分指标: {tuner.scoring}")
print(f"交叉验证折数: {tuner.cv}")

超参数优化器初始化完成
模型类型: gradient_boosting
评分指标: f1_weighted
交叉验证折数: 5


## 方法1：网格搜索（Grid Search）

网格搜索会遍历所有可能的参数组合，找到最优参数。

**优点**：能找到全局最优解（在给定的参数空间内）  
**缺点**：计算成本高，参数组合数呈指数增长  
**适用场景**：参数空间较小，计算资源充足

In [13]:
# # 方法1：网格搜索
# print("\n" + "="*60)
# print("方法1：网格搜索（Grid Search）")
# print("="*60)

# # 创建基础模型
# from sklearn.ensemble import GradientBoostingClassifier
# from src.core.hyperparameter_tuning import HyperparameterTuner
# base_model_grid = GradientBoostingClassifier(random_state=42)

# tuner=HyperparameterTuner(
#     model_type='gradient_boosting',
#     task='classification',
#     cv=5,)
# # 执行网格搜索
# best_params_grid = tuner.grid_search(
#     model=base_model_grid,
#     X_train=X_train,
#     y_train=y_train
# )

# print("\n网格搜索结果:")
# print(f"最佳参数: {best_params_grid}")
# print(f"最佳得分: {tuner.best_score_:.4f}")

# # 显示前10个最佳参数组合
# print("\n前10个最佳参数组合:")
# summary_grid = tuner.get_search_results_summary()
# print(summary_grid.head(10).to_string())

## 使用最佳参数重新训练模型

使用搜索到的最佳参数重新训练模型，并评估性能提升。

In [14]:
# # 使用最佳参数重新训练模型
# print("\n" + "="*60)
# print("使用最佳参数重新训练模型")
# print("="*60)
# best_params_final = best_params_grid
# best_tuner = tuner
# # 选择最佳方法的参数
# # if best_method == 'Grid Search':
# #     best_params_final = best_params_grid
# #     best_tuner = tuner
# # elif best_method == 'Random Search':
# #     best_params_final = best_params_random
# #     best_tuner = tuner_random
# # else:
# #     best_params_final = best_params_bayes
# #     best_tuner = tuner_bayes

# # print(f"\n使用 {best_method} 的最佳参数:")
# for param, value in best_params_final.items():
#     print(f"  {param}: {value}")

# # 使用最佳参数创建新模型
# model_optimized = MLModel(model_type='gradient_boosting', task='classification')

# # 使用最佳参数训练
# print("\n训练优化后的模型...")
# model_optimized.model = GradientBoostingClassifier(**best_params_final, random_state=42)
# model_optimized.model.fit(X_train, y_train)

# # 评估优化后的模型
# metrics_optimized = model_optimized.evaluate(X_test, y_test)

# print("\n优化后模型性能:")
# print(f"  准确率: {metrics_optimized.get('accuracy', 0):.4f}")
# print(f"  F1分数: {metrics_optimized.get('f1', 0):.4f}")
# print(f"  精确率: {metrics_optimized.get('precision', 0):.4f}")
# print(f"  召回率: {metrics_optimized.get('recall', 0):.4f}")

# # 与原始模型比较
# print("\n性能对比:")
# print(f"  原始模型 F1: {metrics.get('f1', 0):.4f}")
# print(f"  优化模型 F1: {metrics_optimized.get('f1', 0):.4f}")
# improvement = (metrics_optimized.get('f1', 0) - metrics.get('f1', 0)) / metrics.get('f1', 1) * 100
# print(f"  提升幅度: {improvement:+.2f}%")

# # 保存优化结果
# best_tuner.save_results('models/tuning_results')
# print("\n✓ 优化结果已保存到 models/tuning_results/")

# # 更新全局model变量为优化后的模型
# model = model_optimized
# print("\n✓ 全局模型已更新为优化后的模型")

In [ ]:
signal_generator = SignalGenerator(model, threshold=0.5, signal_holding_days=20)
signals = signal_generator.generate_signals(X_test, use_probability=True)
signals.index = test_idx
print(f"    ✓ 生成 {len(signals)} 个信号")
print(f"    信号分布: {signals.value_counts().to_dict()}")

# 获取价差价格数据
print("  [5.2] 准备价格数据...")
# 确保spread_data索引与test_idx时区一致
spread_df_for_backtest = spread_data['Coking_Profit'].copy()
if hasattr(spread_df_for_backtest.index, 'tz') and spread_df_for_backtest.index.tz is not None:
    spread_df_for_backtest.index = spread_df_for_backtest.index.tz_localize(None)

spread_prices = spread_df_for_backtest.loc[test_idx, ['spread']].copy()
spread_prices.columns = ['close']
spread_prices['volatility'] = spread_prices['close'].pct_change().rolling(20).std()
print(f"    ✓ 价格数据: {len(spread_prices)} 条")

# 运行回测
print("  [5.3] 运行回测...")
backtest_engine = BacktestEngine(
    initial_capital=1000000,
    commission_rate=0.0005,
    slippage_rate=0.0001,
    max_position=1e16,
    max_capital_usage=0.05
)

equity_curve = backtest_engine.run_backtest(
    spread_prices,
    signals,
    price_col='close',
    volatility_col='volatility'
)
print(f"    ✓ 回测完成，最终权益: ${equity_curve['equity'].iloc[-1]:,.2f}")

# 获取交易日志
trade_log = backtest_engine.get_trade_log()
print(f"    ✓ 总交易次数: {len(trade_log)}")

# 绩效分析
print("  [5.4] 绩效分析...")
analyzer = PerformanceAnalyzer(
    equity_curve,
    initial_capital=1000000,
    risk_free_rate=0.02
)

performance_report = analyzer.generate_performance_report(trade_log)
print(f"    ✓ 总收益率: {performance_report.get('total_return', 0)*100:.2f}%")
print(f"    ✓ 夏普比率: {performance_report.get('sharpe_ratio', 0):.2f}")
print(f"    ✓ 最大回撤: {performance_report.get('max_drawdown', 0)*100:.2f}%")

print("✓ 回测完成")
logger.info("回测完成\n")

# 🔍 调试点6：在此处设置断点，检查回测结果
# 可以查看: equity_curve.tail(), trade_log.head(), performance_report

# ============================================================
# 步骤6：可视化
# ============================================================
print("\n[7/9] 开始可视化...")
logger.info("\n" + "="*60)
logger.info("步骤6：结果可视化")
logger.info("="*60)

print("  [6.1] 生成价格和价差图...")
visualizer.plot_price_and_spread(
    price_data,
    spread_data['Coking_Profit'],
    title='Coking_Profit'
)
print("    ✓ price_spread_chart.png")

print("  [6.2] 生成权益曲线图...")
visualizer.plot_equity_curve(equity_curve)
print("    ✓ equity_curve.png")

print("  [6.3] 生成收益率分布图...")
returns = equity_curve['equity'].pct_change().dropna()
visualizer.plot_returns_distribution(returns)
print("    ✓ returns_distribution.png")

print("  [6.4] 生成月度收益热力图...")
visualizer.plot_monthly_returns_heatmap(equity_curve)
print("    ✓ monthly_returns_heatmap.png")

print("  [6.5] 生成滚动指标图...")
visualizer.plot_rolling_metrics(equity_curve, window=60)
print("    ✓ rolling_metrics.png")

print("  [6.6] 生成交易分析图...")
visualizer.plot_trade_analysis(trade_log)
print("    ✓ trade_analysis.png")

print("✓ 可视化完成")
logger.info("可视化完成\n")

INFO:src.core.ml_models:使用固定阈值模式: 上阈值=0.05, 下阈值=-0.05
INFO:src.core.ml_models:回归信号统计:
INFO:src.core.ml_models:  预测值范围: [-9.5793, 3.0636]
INFO:src.core.ml_models:  做多信号(1): 195 (20.9%)
INFO:src.core.ml_models:  观望信号(0): 26 (2.8%)


INFO:src.core.ml_models:  做空信号(-1): 711 (76.3%)
INFO:src.core.ml_models:生成交易信号完成，信号分布:
INFO:src.core.ml_models:-1    711
 1    195
 0     26
Name: signal, dtype: int64
INFO:src.core.ml_models:应用20天信号维持后，信号分布:
INFO:src.core.ml_models:-1    731
 1    201
Name: signal, dtype: int64


    ✓ 生成 932 个信号
    信号分布: {(-9.579294184083587, -1, 1.0): 1, (-0.3533807406547973, -1, 1.0): 1, (-0.3794884997541652, -1, 1.0): 1, (-0.3792486093420331, -1, 1.0): 1, (-0.37306102171412014, -1, 1.0): 1, (-0.37295935528537677, -1, 1.0): 1, (-0.3659872277123976, -1, 1.0): 1, (-0.3638887545527088, -1, 1.0): 1, (-0.3617917431521986, -1, 1.0): 1, (-0.360657887313367, -1, 1.0): 1, (-0.3595096008623464, -1, 1.0): 1, (-0.3589449011858663, -1, 1.0): 1, (-0.35791861404727077, -1, 1.0): 1, (-0.3578565365048274, -1, 1.0): 1, (-0.34881432632859416, -1, 1.0): 1, (-0.4732404051141878, -1, 1.0): 1, (-0.3476050941622225, -1, 1.0): 1, (-0.34580711459260105, -1, 1.0): 1, (-0.34180902670569624, -1, 1.0): 1, (-0.334779758268477, -1, 1.0): 1, (-0.3330340022615893, -1, 1.0): 1, (-0.33282257783902486, -1, 1.0): 1, (-0.32847576189163796, -1, 1.0): 1, (-0.3270432594067775, -1, 1.0): 1, (-0.3251248135870263, -1, 1.0): 1, (-0.32098833133865545, -1, 1.0): 1, (-0.31729198002146547, -1, 1.0): 1, (-0.3039351806575386

KeyError: "None of [DatetimeIndex(['2022-02-03 05:00:00', '2022-02-04 05:00:00',\n               '2022-02-07 05:00:00', '2022-02-08 05:00:00',\n               '2022-02-09 05:00:00', '2022-02-10 05:00:00',\n               '2022-02-11 05:00:00', '2022-02-14 05:00:00',\n               '2022-02-15 05:00:00', '2022-02-16 05:00:00',\n               ...\n               '2025-10-06 04:00:00', '2025-10-07 04:00:00',\n               '2025-10-08 04:00:00', '2025-10-09 04:00:00',\n               '2025-10-10 04:00:00', '2025-10-13 04:00:00',\n               '2025-10-14 04:00:00', '2025-10-15 04:00:00',\n               '2025-10-16 04:00:00', '2025-10-17 04:00:00'],\n              dtype='datetime64[ns]', name='date', length=932, freq=None)] are in the [index]"